Теперь после всего так как у меня уже есть код обучения моделей RandomForest, XGBoost, Lightgbm, мне нужно чтобы ты его просматрел. Смотри мне нужно поменять обучение моделей, и можно ли как то сохранить выводы для анализа или они не подходят к новым моделям?

Вот код обучения моделей где RandomForest, XGBoost, LightGBM (не валидный для временных рядов модели) ==  
Импорт библиотек
import pandas as pd
import numpy as np
import os
import logging
import warnings

import joblib
import xgboost as xgb
import matplotlib.pyplot as plt

from lightgbm import LGBMRegressor
from sklearn.model_selection import train_test_split
from sklearn.ensemble import RandomForestRegressor, VotingRegressor
from sklearn.metrics import mean_absolute_error, mean_squared_error, r2_score

# Отключаем логи LightGBM
logging.getLogger('lightgbm').setLevel(logging.WARNING)

os.makedirs('models', exist_ok=True)

print("Библиотеки загружены!")
Библиотеки загружены!
# УЛУЧШЕННОЕ СОЗДАНИЕ ПРИЗНАКОВ ПОЛНОСТЬЮ БЕЗ УТЕЧЕК
print("Создание УЛУЧШЕННЫХ признаков ПОЛНОСТЬЮ БЕЗ УТЕЧЕК...")
df = pd.read_csv('df/obr.csv', parse_dates=['datetime'], index_col='datetime')
print(f"Исходный размер: {df.shape}")

# ===== ОСНОВНЫЕ ВРЕМЕННЫЕ ПРИЗНАКИ (БЕЗОПАСНЫЕ) =====
# Циклические признаки - ✅ БЕЗОПАСНО
df['hour_sin'] = np.sin(2 * np.pi * df['hour']/24)
df['hour_cos'] = np.cos(2 * np.pi * df['hour']/24)
df['month_sin'] = np.sin(2 * np.pi * df['month']/12)
df['month_cos'] = np.cos(2 * np.pi * df['month']/12)
df['day_of_week_sin'] = np.sin(2 * np.pi * df['day_of_week']/7)
df['day_of_week_cos'] = np.cos(2 * np.pi * df['day_of_week']/7)
df['day_of_year'] = df.index.dayofyear
df['day_of_year_sin'] = np.sin(2 * np.pi * df['day_of_year']/365)
df['day_of_year_cos'] = np.cos(2 * np.pi * df['day_of_year']/365)

# ===== ВРЕМЕННЫЕ ПАТТЕРНЫ (БЕЗОПАСНЫЕ) =====
# СУТОЧНЫЕ ПАТТЕРНЫ
df['is_early_morning'] = ((df['hour'] >= 4) & (df['hour'] <= 6)).astype(int)
df['is_midday'] = ((df['hour'] >= 10) & (df['hour'] <= 16)).astype(int)
df['is_late_evening'] = ((df['hour'] >= 21) & (df['hour'] <= 23)).astype(int)
# df['is_evening_peak'] = ((df['hour'] >= 18) & (df['hour'] <= 22)).astype(int)
# df['is_morning_peak'] = ((df['hour'] >= 7) & (df['hour'] <= 9)).astype(int)
df['is_night'] = ((df['hour'] >= 0) & (df['hour'] <= 5)).astype(int)
df['is_deep_night'] = ((df['hour'] >= 1) & (df['hour'] <= 4)).astype(int)

# НЕДЕЛЬНЫЕ ПАТТЕРНЫ
df['is_monday'] = (df['day_of_week'] == 0).astype(int)
df['is_friday'] = (df['day_of_week'] == 4).astype(int)
df['is_sunday'] = (df['day_of_week'] == 6).astype(int)
df['is_week_start'] = (df['day_of_week'].isin([0, 1])).astype(int)
df['is_week_end'] = (df['day_of_week'].isin([4, 5])).astype(int)
df['is_family_time'] = ((df['hour'] >= 18) & (df['hour'] <= 22) & (df['is_weekend'] == 0)).astype(int)

# СЕЗОННЫЕ ПАТТЕРНЫ
df['is_high_season'] = df['month'].isin([1, 2, 12]).astype(int)
df['is_low_season'] = df['month'].isin([6, 7, 8]).astype(int)
df['is_spring'] = df['month'].isin([3, 4, 5]).astype(int)

# КРИТИЧЕСКИЕ ПЕРИОДЫ
df['morning_surge_6_7'] = ((df['hour'] >= 6) & (df['hour'] <= 7)).astype(int)
df['evening_surge_17_18'] = ((df['hour'] >= 17) & (df['hour'] <= 18)).astype(int)
df['evening_drop_22_23'] = ((df['hour'] >= 22) & (df['hour'] <= 23)).astype(int)

# ВЗАИМОДЕЙСТВИЯ ВРЕМЕНИ
df['winter_evening'] = (df['is_high_season'] & df['is_evening_peak']).astype(int)
df['summer_afternoon'] = (df['is_low_season'] & df['is_midday']).astype(int)
df['workday_evening'] = ((~df['is_weekend']) & df['is_evening_peak']).astype(int)
df['sunday_evening'] = (df['is_sunday'] & df['is_evening_peak']).astype(int)
df['weekend_evening_boost'] = (df['is_weekend'] & df['is_evening_peak']).astype(int)
df['weekend_morning'] = (df['is_weekend'] & df['is_morning_peak']).astype(int)
df['winter_weekend_evening'] = (df['is_high_season'] & df['is_weekend'] & df['is_evening_peak']).astype(int)

# ===== ЛАГИ ЦЕЛЕВОЙ ПЕРЕМЕННОЙ (БЕЗОПАСНЫЕ) =====
print("Создание лагов на основе анализа автокорреляции...")
df['lag_1h'] = df['Global_active_power'].shift(1)    # 1 час назад
df['lag_2h'] = df['Global_active_power'].shift(2)    # 2 часа назад
df['lag_3h'] = df['Global_active_power'].shift(3)    # 3 часа назад
df['lag_6h'] = df['Global_active_power'].shift(6)    # 6 часов назад
df['lag_12h'] = df['Global_active_power'].shift(12)  # 12 часов назад
df['lag_24h'] = df['Global_active_power'].shift(24)  # 1 день назад
df['lag_48h'] = df['Global_active_power'].shift(48)  # 2 дня назад  
df['lag_72h'] = df['Global_active_power'].shift(72)  # 3 дня назад
df['lag_168h'] = df['Global_active_power'].shift(168)  # 1 неделя назад

# ===== СКОЛЬЗЯЩИЕ СТАТИСТИКИ (БЕЗОПАСНЫЕ) =====
print("Создание скользящих статистик БЕЗ утечек...")
# Двойной сдвиг для полной безопасности
df['rolling_mean_6h'] = df['Global_active_power'].shift(1).rolling(6, min_periods=1).mean()
df['rolling_mean_12h'] = df['Global_active_power'].shift(1).rolling(12, min_periods=1).mean()
df['rolling_mean_24h'] = df['Global_active_power'].shift(1).rolling(24, min_periods=1).mean()
df['rolling_mean_72h'] = df['Global_active_power'].shift(1).rolling(72, min_periods=1).mean()
df['rolling_mean_168h'] = df['Global_active_power'].shift(1).rolling(168, min_periods=1).mean()
df['rolling_mean_720h'] = df['Global_active_power'].shift(1).rolling(720, min_periods=1).mean()

df['rolling_std_6h'] = df['Global_active_power'].shift(1).rolling(6, min_periods=1).std()
df['rolling_std_12h'] = df['Global_active_power'].shift(1).rolling(12, min_periods=1).std()
df['rolling_std_24h'] = df['Global_active_power'].shift(1).rolling(24, min_periods=1).std()
df['rolling_std_72h'] = df['Global_active_power'].shift(1).rolling(72, min_periods=1).std()
df['rolling_std_168h'] = df['Global_active_power'].shift(1).rolling(168, min_periods=1).std()

# ===== ПРОИЗВОДНЫЕ ОТ ЛАГОВ (БЕЗОПАСНЫЕ) =====
print("Создание производных от лагов...")
# df['momentum_1h'] = df['lag_1h'] - df['Global_active_power'].shift(2)  # Тренд за 1 час
df['momentum_1h'] = df['Global_active_power'].shift(1) - df['Global_active_power'].shift(2)
# df['momentum_3h'] = df['lag_3h'] - df['Global_active_power'].shift(6)  # Тренд за 3 часа
# df['momentum_3h'] = df['lag_3h'] - df['lag_6h']
# df['momentum_24h'] = df['lag_24h'] - df['Global_active_power'].shift(48)  # Тренд за сутки
# df['momentum_24h'] = df['lag_24h'] - df['lag_48h']

# df['acceleration_1h'] = df['momentum_1h'] - (df['lag_1h'] - df['Global_active_power'].shift(2)).shift(1)
df['acceleration_1h'] = (df['Global_active_power'].shift(1) - 2*df['Global_active_power'].shift(2) + df['Global_active_power'].shift(3))

df['deviation_from_daily_norm'] = df['lag_24h'] - df['rolling_mean_24h']
# df['deviation_from_weekly_norm'] = df['lag_168h'] - df['rolling_mean_168h']

df['volatility_ratio_6h'] = df['rolling_std_6h'] / (df['rolling_mean_6h'] + 0.001)
df['volatility_ratio_24h'] = df['rolling_std_24h'] / (df['rolling_mean_24h'] + 0.001)

# df['weekly_seasonal_factor'] = df['lag_168h'] / (df['rolling_mean_168h'] + 0.001)
# df['weekly_seasonal_factor'] = df['lag_168h'] / (df['rolling_mean_168h'].shift(167) + 0.001)
df['daily_seasonal_factor'] = df['lag_24h'] / (df['rolling_mean_24h'] + 0.001)

# ===== ФИЗИЧЕСКИЕ ПРИЗНАКИ (БЕЗОПАСНЫЕ) =====
# print("Создание физических признаков...")
# df['load_behavior_lag_1h'] = df['Global_intensity'].shift(1) * df['Voltage'].shift(1) / 1000
# df['power_factor_estimate'] = df['Global_active_power'].shift(1) / (df['load_behavior_lag_1h'] + 0.001)

# ===== ВЗАИМОДЕЙСТВИЯ ЛАГОВ С ВРЕМЕНЕМ (БЕЗОПАСНЫЕ) =====
print("Создание взаимодействий лагов...")
df['lag_1h_morning_boost'] = df['lag_1h'] * df['is_morning_peak']
df['lag_1h_evening_boost'] = df['lag_1h'] * df['is_evening_peak']
df['lag_24h_weekend_effect'] = df['lag_24h'] * df['is_weekend']
df['lag_168h_seasonal_boost'] = df['lag_168h'] * df['month_sin']

df['momentum_evening_intensity'] = df['momentum_1h'] * df['is_evening_peak']
df['volatility_morning_effect'] = df['rolling_std_24h'] * df['is_morning_peak']

# ===== СЕЗОННЫЕ ВЗАИМОДЕЙСТВИЯ (БЕЗОПАСНЫЕ) =====
print("Создание сезонных взаимодействий...")
df['winter_evening_demand'] = df['is_high_season'] * df['is_evening_peak'] * df['lag_24h']
df['summer_afternoon_cooling'] = df['is_low_season'] * df['is_midday'] * df['rolling_mean_24h']  # Безопасная замена
df['spring_transition_effect'] = df['is_spring'] * df['day_of_year_sin'] * df['rolling_std_24h']

df['seasonal_volatility'] = df['day_of_year_sin'] * df['rolling_std_24h']
df['weekly_seasonal_intensity'] = df['lag_168h'] * df['is_weekend'] * df['rolling_mean_24h']

# ===== ПОВЕДЕНЧЕСКИЕ ПАТТЕРНЫ (БЕЗОПАСНЫЕ) =====
print("Создание поведенческих паттернов...")
df['evening_peak_intensity_safe'] = df['is_evening_peak'] * df['rolling_std_24h'] * df['rolling_mean_24h']  # Безопасная замена
df['morning_peak_momentum_safe'] = df['is_morning_peak'] * df['momentum_1h']
df['weekend_lifestyle_safe'] = df['is_weekend'] * df['rolling_mean_24h'] * df['rolling_std_24h']  # Безопасная замена
df['workday_routine_safe'] = (~df['is_weekend'].astype(bool)).astype(int) * df['rolling_std_24h'] * df['rolling_mean_24h']  # Безопасная замена

# ===== ДОПОЛНИТЕЛЬНЫЕ БЕЗОПАСНЫЕ ПРИЗНАКИ =====
print("Создание дополнительных безопасных признаков...")
# Паттерны пиковых периодов
df['peak_morning_7_9'] = ((df['hour'] >= 7) & (df['hour'] <= 9)).astype(int)
df['peak_evening_18_22'] = ((df['hour'] >= 18) & (df['hour'] <= 22)).astype(int)

df['morning_intensity_safe'] = df['peak_morning_7_9'] * df['rolling_std_24h'] * df['lag_1h']
df['evening_intensity_safe'] = df['peak_evening_18_22'] * df['rolling_std_24h'] * df['lag_24h']

# Критические переходы
df['surge_6_7_safe'] = df['morning_surge_6_7'] * df['momentum_1h']
df['drop_22_23_safe'] = df['evening_drop_22_23'] * df['rolling_std_24h']

# Сложные сезонные взаимодействия
df['seasonal_energy_demand_safe'] = df['day_of_year_sin'] * df['month_sin'] * df['rolling_std_24h'] * df['rolling_mean_24h']  # Безопасная замена
df['weather_behavior_safe'] = df['day_of_year_cos'] * df['is_high_season'] * df['lag_24h']
df['monthly_energy_cycle_safe'] = df['month_sin'] * df['rolling_mean_168h'] * df['rolling_std_24h']  # Безопасная замена

df['seasonal_peak_intensity_safe'] = (
    df['is_evening_peak'] * 
    df['day_of_year_sin'] * 
    (df['is_high_season'] * 1.5 + df['is_low_season'] * 0.7)
)

df['long_term_seasonal_safe'] = df['day_of_year_sin'] * df['rolling_mean_720h']

# ===== ФИНАЛЬНЫЕ ПРОИЗВОДНЫЕ (БЕЗОПАСНЫЕ) =====
print("Создание финальных производных признаков...")
df['composite_energy_score'] = (
    df['lag_1h'] * 0.3 +
    df['momentum_1h'] * 0.2 + 
    df['rolling_mean_24h'] * 0.2 +
    df['rolling_std_24h'] * 0.15 +
    df['day_of_year_sin'] * 0.15
)

df['behavioral_energy_pattern'] = (
    df['is_evening_peak'] * df['lag_1h'] * 0.4 +
    df['is_weekend'] * df['rolling_mean_24h'] * 0.3 +
    df['is_high_season'] * df['rolling_std_24h'] * 0.3
)

# ===== ОБРАБОТКА ПРОПУСКОВ =====
print("Обработка пропусков...")
initial_size = len(df)

# Заполняем пропуски в числовых признаках медианой
numeric_columns = df.select_dtypes(include=[np.number]).columns
for col in numeric_columns:
    if df[col].isnull().any():
        df[col] = df[col].fillna(df[col].median())

# Удаляем оставшиеся пропуски (если есть)
df = df.dropna()

print(f"Признаки созданы. Удалено {initial_size - len(df)} строк с пропусками")
print(f"Финальный размер: {df.shape}")
print(f"Создано признаков: {len([col for col in df.columns if col not in ['Global_active_power', 'datetime']])}")

# ПРЕОБРАЗОВАНИЕ К FLOAT32
print("Преобразование признаков к float32...")
for col in df.columns:
    if col != 'Global_active_power':
        df[col] = df[col].astype('float32')

print("✅ Все признаки созданы ПОЛНОСТЬЮ БЕЗ УТЕЧЕК!")
Создание УЛУЧШЕННЫХ признаков ПОЛНОСТЬЮ БЕЗ УТЕЧЕК...
Исходный размер: (260640, 14)
Создание лагов на основе анализа автокорреляции...
Создание скользящих статистик БЕЗ утечек...
Создание производных от лагов...
Создание взаимодействий лагов...
Создание сезонных взаимодействий...
Создание поведенческих паттернов...
Создание дополнительных безопасных признаков...
Создание финальных производных признаков...
Обработка пропусков...
Признаки созданы. Удалено 0 строк с пропусками
Финальный размер: (260640, 100)
Создано признаков: 99
Преобразование признаков к float32...
✅ Все признаки созданы ПОЛНОСТЬЮ БЕЗ УТЕЧЕК!
# УДАЛИТЬ ВСЕ ПРИЗНАКИ, ИСПОЛЬЗУЮЩИЕ ПРОБЛЕМНЫЕ ПЕРЕМЕННЫЕ
leaky_features = [
    'winter_evening', 'workday_evening', 'sunday_evening', 
    'weekend_evening_boost', 'weekend_morning', 'winter_weekend_evening',
    'lag_1h_evening_boost', 'momentum_evening_intensity', 'volatility_morning_effect',
    'winter_evening_demand', 'evening_peak_intensity_safe', 'morning_peak_momentum_safe',
    'seasonal_peak_intensity_safe'
]

# Удаляем из DataFrame
df = df.drop(columns=[f for f in leaky_features if f in df.columns])
print(f"Удалено {len(leaky_features)} признаков с утечками")


# ЗАМЕНИТЬ проблемные признаки на безопасные аналоги
df['evening_hours'] = ((df['hour'] >= 18) & (df['hour'] <= 22)).astype(int)
df['morning_hours'] = ((df['hour'] >= 7) & (df['hour'] <= 9)).astype(int)

# Безопасные версии взаимодействий
df['lag_1h_evening_safe'] = df['lag_1h'] * df['evening_hours']
df['momentum_evening_safe'] = df['momentum_1h'] * df['evening_hours']
Удалено 13 признаков с утечками
df.sample(5)
Global_active_power	Global_reactive_power	Voltage	Global_intensity	Sub_metering_1	Sub_metering_2	Sub_metering_3	hour	day_of_week	month	...	seasonal_energy_demand_safe	weather_behavior_safe	monthly_energy_cycle_safe	long_term_seasonal_safe	composite_energy_score	behavioral_energy_pattern	evening_hours	morning_hours	lag_1h_evening_safe	momentum_evening_safe
datetime																					
2007-06-28 20:37:00	4.526	0.000	236.860001	19.200001	0.0	31.0	17.0	20.0	3.0	6.0	...	1.186280e-17	-0.000000	1.416197e-16	0.070995	1.300404	1.012000	1	0	2.53	-0.87
2007-05-22 02:57:00	1.236	0.064	235.910004	5.200000	0.0	0.0	18.0	2.0	1.0	5.0	...	1.065291e-01	-0.000000	8.748428e-02	0.475748	0.679067	0.000000	0	0	0.00	-0.00
2007-02-06 12:40:00	0.392	0.000	245.669998	1.800000	0.0	0.0	0.0	12.0	1.0	2.0	...	1.637215e-01	0.381062	2.951882e-01	0.433030	0.416850	0.132081	0	0	0.00	-0.00
2007-06-14 04:00:00	0.180	0.100	241.330002	0.800000	0.0	1.0	0.0	4.0	3.0	6.0	...	3.221939e-19	-0.000000	9.062647e-19	0.175952	0.152031	0.000000	0	0	0.00	0.00
2007-01-29 10:33:00	1.346	0.086	236.869995	5.600000	0.0	0.0	17.0	10.0	0.0	1.0	...	8.883254e-02	1.617203	2.022903e-01	0.366199	0.821976	0.070431	0	0	0.00	-0.00
5 rows × 91 columns

# ОБНОВЛЕННЫЙ СПИСОК ПРИЗНАКОВ (без удаленных)
optimal_features_safe_updated = [
    # ЦИКЛИЧЕСКИЕ ПРИЗНАКИ
    'hour_sin', 'hour_cos', 'month_sin', 'day_of_year_sin', 'day_of_year_cos',
    
    # ВРЕМЕННЫЕ ПАТТЕРНЫ  
    'is_weekend', 'is_family_time', 'morning_surge_6_7',
    
    # ЛАГИ
    'lag_1h', 'lag_24h', 'lag_168h',
    
    # СКОЛЬЗЯЩИЕ СТАТИСТИКИ
    'rolling_mean_6h', 'rolling_mean_24h', 'rolling_mean_720h',
    'rolling_std_6h', 'rolling_std_24h',
    
    # ПРОИЗВОДНЫЕ
    'momentum_1h', 'acceleration_1h', 'deviation_from_daily_norm', 'volatility_ratio_6h',
    
    # БЕЗОПАСНЫЕ ВЗАИМОДЕЙСТВИЯ
    'lag_24h_weekend_effect', 'lag_168h_seasonal_boost',
    
    # СЕЗОННЫЕ ПАТТЕРНЫ
    'spring_transition_effect', 'seasonal_volatility',
    
    # КРИТИЧЕСКИЕ ПЕРИОДЫ  
    'surge_6_7_safe', 'evening_intensity_safe',
    
    # СЕЗОННЫЕ ПРОИЗВОДНЫЕ
    'weather_behavior_safe', 'long_term_seasonal_safe',
    
    # ФИНАЛЬНЫЕ КОМПОЗИТЫ
    'composite_energy_score',
    
    # НОВЫЕ БЕЗОПАСНЫЕ ПРИЗНАКИ
    'evening_hours', 'morning_hours', 'lag_1h_evening_safe', 'momentum_evening_safe'
]

# ФИЛЬТРУЕМ ТОЛЬКО СУЩЕСТВУЮЩИЕ ПРИЗНАКИ
available_features = [f for f in optimal_features_safe_updated if f in df.columns]
print(f"Используется {len(available_features)} безопасных признаков")


# ===== ИСПРАВЛЕННОЕ РАЗДЕЛЕНИЕ ДАННЫХ =====
print("Подготовка данных БЕЗ утечек...")

# Подготовка данных
df_safe = df[available_features + ['Global_active_power']].dropna()
X = df_safe[available_features]
y = df_safe['Global_active_power']

print(f"Доступно данных после обработки: {len(df_safe):,} записей")

# ИСПРАВЛЕННОЕ ВРЕМЕННОЕ РАЗДЕЛЕНИЕ:
split_date = '2007-04-15'  # Train: Jan-Mar
buffer_end = '2007-04-29'  # Начало тестовой выборки

# ВАЖНО: использовать ТОЛЬКО df_safe.index для всех выборок!
train_mask = df_safe.index < split_date
test_mask = df_safe.index >= buffer_end

X_train = X[train_mask]
X_test = X[test_mask]
y_train = y[train_mask] 
y_test = y[test_mask]

print(f"Обучающая выборка: {X_train.shape[0]:,} записей ({X_train.index.min()} - {X_train.index.max()})")
print(f"Тестовая выборка: {X_test.shape[0]:,} записей ({X_test.index.min()} - {X_test.index.max()})")
print(f"Размерность признаков: {X_train.shape[1]}")

# Проверка согласованности размеров
print(f"\n=== ПРОВЕРКА РАЗМЕРОВ ===")
print(f"X_train: {X_train.shape}, y_train: {y_train.shape}")
print(f"X_test: {X_test.shape}, y_test: {y_test.shape}")

if X_train.shape[0] == y_train.shape[0] and X_test.shape[0] == y_test.shape[0]:
    print("✅ Размеры согласованы!")
else:
    print("❌ ОШИБКА: Размеры не согласованы!")
    # Автоматическое исправление
    common_train_idx = X_train.index.intersection(y_train.index)
    common_test_idx = X_test.index.intersection(y_test.index)
    
    X_train = X_train.loc[common_train_idx]
    y_train = y_train.loc[common_train_idx]
    X_test = X_test.loc[common_test_idx] 
    y_test = y_test.loc[common_test_idx]
    
    print(f"Исправлено: X_train: {X_train.shape}, y_train: {y_train.shape}")

# Анализ распределения целевой переменной
print(f"\n=== СТАТИСТИКА ЦЕЛЕВОЙ ПЕРЕМЕННОЙ ===")
print(f"Среднее потребление (train): {y_train.mean():.3f} кВт")
print(f"Среднее потребление (test):  {y_test.mean():.3f} кВт")
print(f"Std потребление (train): {y_train.std():.3f} кВт")
print(f"Std потребление (test):  {y_test.std():.3f} кВт")

# Проверка временного разрыва
time_gap = X_test.index.min() - X_train.index.max()
print(f"Временной разрыв между train и test: {time_gap}")

print("\n✅ Данные полностью готовы для обучения БЕЗ УТЕЧЕК!")
print("✅ Переменные X_train, X_test, y_train, y_test готовы к использованию!")
Используется 33 безопасных признаков
Подготовка данных БЕЗ утечек...
Доступно данных после обработки: 260,640 записей
Обучающая выборка: 149,760 записей (2007-01-01 00:00:00 - 2007-04-14 23:59:00)
Тестовая выборка: 90,720 записей (2007-04-29 00:00:00 - 2007-06-30 23:59:00)
Размерность признаков: 33

=== ПРОВЕРКА РАЗМЕРОВ ===
X_train: (149760, 33), y_train: (149760,)
X_test: (90720, 33), y_test: (90720,)
✅ Размеры согласованы!

=== СТАТИСТИКА ЦЕЛЕВОЙ ПЕРЕМЕННОЙ ===
Среднее потребление (train): 1.360 кВт
Среднее потребление (test):  0.898 кВт
Std потребление (train): 1.276 кВт
Std потребление (test):  0.972 кВт
Временной разрыв между train и test: 14 days 00:01:00

✅ Данные полностью готовы для обучения БЕЗ УТЕЧЕК!
✅ Переменные X_train, X_test, y_train, y_test готовы к использованию!
# ОБУЧЕНИЕ МОДЕЛЕЙ
print("Обучение моделей...")

# Словарь для хранения моделей и результатов
models = {}
results = {}

# A. RANDOM FOREST
print("\nОбучение RandomForest...")
rf_model = RandomForestRegressor(
    n_estimators=100,
    max_depth=20,
    random_state=42,
    n_jobs=-1
)
rf_model.fit(X_train, y_train)
models['RandomForest'] = rf_model

# B. XGBOOST
print("Обучение XGBoost...")
xgb_model = xgb.XGBRegressor(
    n_estimators=100,
    max_depth=10,
    learning_rate=0.1,
    random_state=42,
    n_jobs=-1
)
xgb_model.fit(X_train, y_train)
models['XGBoost'] = xgb_model

# C. LIGHTGBM
print("Обучение LightGBM...")
lgb_model = LGBMRegressor(
    n_estimators=100,
    max_depth=10,
    learning_rate=0.1,
    random_state=42,
    n_jobs=-1,
    verbose=-1
)
lgb_model.fit(X_train, y_train)
models['LightGBM'] = lgb_model

print("Все модели обучены!")
Обучение моделей...

Обучение RandomForest...
Обучение XGBoost...
Обучение LightGBM...
Все модели обучены!
# ОЦЕНКА КАЧЕСТВА МОДЕЛЕЙ
print("\nОценка качества моделей на тестовой выборке:")

for name, model in models.items():
    # Прогнозы
    y_pred = model.predict(X_test)
    
    # Метрики
    mae = mean_absolute_error(y_test, y_pred)
    rmse = np.sqrt(mean_squared_error(y_test, y_pred))
    r2 = r2_score(y_test, y_pred)
    
    results[name] = {
        'MAE': mae,
        'RMSE': rmse, 
        'R2': r2,
        'predictions': y_pred
    }
    
    print(f"\n{name}:")
    print(f"  MAE:  {mae:.4f} кВт")
    print(f"  RMSE: {rmse:.4f} кВт") 
    print(f"  R²:   {r2:.4f}")

# Сравнение моделей
print("\nСРАВНЕНИЕ МОДЕЛЕЙ:")
best_model = min(results, key=lambda x: results[x]['MAE'])
print(f"Лучшая модель по MAE: {best_model}")

# ВИЗУАЛИЗАЦИЯ РЕЗУЛЬТАТОВ
print("\nВизуализация результатов...")

# ГРАФИК СРАВНЕНИЯ МЕТРИК
plt.figure(figsize=(15, 10))

# График 1: Сравнение метрик
plt.subplot(2, 2, 1)
metrics = ['MAE', 'RMSE']
models_names = list(results.keys())
mae_values = [results[model]['MAE'] for model in models_names]
rmse_values = [results[model]['RMSE'] for model in models_names]

x = np.arange(len(models_names))
width = 0.35

plt.bar(x - width/2, mae_values, width, label='MAE', alpha=0.8, color='skyblue')
plt.bar(x + width/2, rmse_values, width, label='RMSE', alpha=0.8, color='lightcoral')

plt.xlabel('Модели')
plt.ylabel('Ошибка (кВт)')
plt.title('Сравнение MAE и RMSE моделей')
plt.xticks(x, models_names)
plt.legend()
plt.grid(True, alpha=0.3)

# График 2: R² сравнение (ИСПРАВЛЕННЫЙ)
plt.subplot(2, 2, 2)
r2_values = [results[model]['R2'] for model in models_names]
colors = ['green' if model == best_model else 'gray' for model in models_names]

# Автоматически подбираем масштаб
r2_min, r2_max = min(r2_values), max(r2_values)
plt.ylim(r2_min - 0.001, r2_max + 0.001)  # Динамический масштаб

plt.bar(models_names, r2_values, color=colors, alpha=0.7)
plt.xlabel('Модели')
plt.ylabel('R²')
plt.title('Коэффициент детерминации R²')
plt.grid(True, alpha=0.3)

# Добавляем значения на столбцы
for i, v in enumerate(r2_values):
    plt.text(i, v + 0.0001, f'{v:.4f}', ha='center', va='bottom', fontsize=9)

# График 3: Прогнозы vs Факт (для лучшей модели)
plt.subplot(2, 2, 3)
best_predictions = results[best_model]['predictions']

# # Берем первые 100 точек для наглядности
# sample_size = min(100, len(y_test))
# plt.plot(y_test.values[:sample_size], label='Факт', marker='o', markersize=3)
# plt.plot(best_predictions[:sample_size], label='Прогноз', marker='x', markersize=3)
# plt.xlabel('Временные точки')
# plt.ylabel('Нагрузка (кВт)')
# plt.title(f'Прогноз vs Факт ({best_model}) - первые 100 точек')
# plt.legend()
# plt.grid(True, alpha=0.3)

# СДЕЛАЙ ТАК:
sample_indices = np.random.choice(len(y_test), size=min(100, len(y_test)), replace=False)
sample_indices = np.sort(sample_indices)  # для красивого графика

plt.plot(y_test.values[sample_indices], label='Факт', marker='o', markersize=3)
plt.plot(best_predictions[sample_indices], label='Прогноз', marker='x', markersize=3)
plt.xlabel('Случайные временные точки')
plt.ylabel('Нагрузка (кВт)')
plt.title(f'Прогноз vs Факт ({best_model}) - случайная выборка 100 точек')

# График 4: Ошибки прогноза
plt.subplot(2, 2, 4)
errors = best_predictions - y_test.values
plt.hist(errors, bins=50, alpha=0.7, color='orange', edgecolor='black')
plt.xlabel('Ошибка прогноза (кВт)')
plt.ylabel('Частота')
plt.title('Распределение ошибок прогноза')
plt.grid(True, alpha=0.3)

plt.tight_layout()
plt.show()

# ТАБЛИЦА С МЕТРИКАМИ
print("\nТАБЛИЦА МЕТРИК:")
print("="*50)
print(f"{'Модель':<15} {'MAE (кВт)':<12} {'RMSE (кВт)':<12} {'R²':<10}")
print("="*50)
for model in models_names:
    mae = results[model]['MAE']
    rmse = results[model]['RMSE']
    r2 = results[model]['R2']
    marker = " " if model == best_model else ""
    print(f"{model:<15} {mae:<12.4f} {rmse:<12.4f} {r2:<10.4f}{marker}")
print("="*50)

# АНАЛИЗ ЛУЧШЕЙ МОДЕЛИ
print(f"\nАНАЛИЗ ЛУЧШЕЙ МОДЕЛИ ({best_model}):")
best_mae = results[best_model]['MAE']
best_rmse = results[best_model]['RMSE']
best_r2 = results[best_model]['R2']

print(f"• Средняя ошибка: {best_mae*1000:.1f} Вт")
print(f"• Точность прогноза: {best_r2*100:.2f}%")
print(f"• Максимальная ошибка: {np.max(np.abs(best_predictions - y_test.values))*1000:.1f} Вт")
Оценка качества моделей на тестовой выборке:

RandomForest:
  MAE:  0.1536 кВт
  RMSE: 0.3108 кВт
  R²:   0.8977

XGBoost:
  MAE:  0.1392 кВт
  RMSE: 0.3054 кВт
  R²:   0.9012

LightGBM:
  MAE:  0.1077 кВт
  RMSE: 0.2821 кВт
  R²:   0.9157

СРАВНЕНИЕ МОДЕЛЕЙ:
Лучшая модель по MAE: LightGBM

Визуализация результатов...

ТАБЛИЦА МЕТРИК:
==================================================
Модель          MAE (кВт)    RMSE (кВт)   R²        
==================================================
RandomForest    0.1536       0.3108       0.8977    
XGBoost         0.1392       0.3054       0.9012    
LightGBM        0.1077       0.2821       0.9157     
==================================================

АНАЛИЗ ЛУЧШЕЙ МОДЕЛИ (LightGBM):
• Средняя ошибка: 107.7 Вт
• Точность прогноза: 91.57%
• Максимальная ошибка: 3776.8 Вт
# ИНТЕРПРЕТАЦИЯ МОДЕЛЕЙ - УЛУЧШЕННАЯ КАТЕГОРИЗАЦИЯ
print("=" * 60)
print("ИНТЕРПРЕТАЦИЯ МОДЕЛЕЙ - АНАЛИЗ ВАЖНОСТИ ПРИЗНАКОВ")
print("=" * 60)

# ВАЖНО: используем ТЕ ЖЕ признаки, что и при обучении моделей!
model_features = X_train.columns.tolist()
print(f"Модели обучены на {len(model_features)} признаках")

# Создаем DataFrame с важностью признаков для всех моделей
feature_importance_df = pd.DataFrame(index=model_features)

# Функция для нормализации важности признаков
def normalize_importance(importance_array):
    """Нормализует важность признаков к диапазону 0-100%"""
    if np.sum(importance_array) == 0:
        return importance_array
    return (importance_array / np.sum(importance_array)) * 100

# Для RandomForest
if hasattr(rf_model, 'feature_importances_'):
    if len(rf_model.feature_importances_) == len(model_features):
        feature_importance_df['RandomForest'] = normalize_importance(rf_model.feature_importances_)

# Для XGBoost - используем gain-based importance
if hasattr(xgb_model, 'get_booster'):
    try:
        xgb_importance = xgb_model.get_booster().get_score(importance_type='gain')
        # Преобразуем словарь в массив в правильном порядке
        xgb_importance_array = np.array([xgb_importance.get(f, 0) for f in model_features])
        feature_importance_df['XGBoost'] = normalize_importance(xgb_importance_array)
    except:
        # Fallback к стандартной важности
        if hasattr(xgb_model, 'feature_importances_'):
            feature_importance_df['XGBoost'] = normalize_importance(xgb_model.feature_importances_)

# Для LightGBM - используем gain-based importance
if hasattr(lgb_model, 'feature_importances_'):
    if len(lgb_model.feature_importances_) == len(model_features):
        feature_importance_df['LightGBM'] = normalize_importance(lgb_model.feature_importances_)

# Проверяем корректность данных
print("\n=== ПРОВЕРКА КОРРЕКТНОСТИ ДАННЫХ ===")
for model in feature_importance_df.columns:
    total_importance = feature_importance_df[model].sum()
    print(f"{model}: сумма важностей = {total_importance:.1f}%")
    
    if abs(total_importance - 100) > 1:  # Допуск 1%
        print(f"⚠️  {model}: возможна ошибка в расчетах!")

# Определяем модель для анализа
if best_model in feature_importance_df.columns:
    sort_column = best_model
elif 'LightGBM' in feature_importance_df.columns:
    sort_column = 'LightGBM'
elif 'XGBoost' in feature_importance_df.columns:
    sort_column = 'XGBoost'
else:
    sort_column = feature_importance_df.columns[0]

# Сортируем по важности
feature_importance_df = feature_importance_df.sort_values(sort_column, ascending=False)

print(f"\nТОП-15 самых важных признаков по версии {sort_column}:")
print("=" * 50)
for i, (feature, importance) in enumerate(feature_importance_df[sort_column].head(15).items()):
    print(f"{i+1:2d}. {feature:<25} {importance:.2f}%")


# ВИЗУАЛИЗАЦИЯ ВАЖНОСТИ ПРИЗНАКОВ
plt.figure(figsize=(16, 12))

# График 1: Важность признаков для лучшей модели
plt.subplot(2, 2, 1)
top_features = feature_importance_df.head(15)
plt.barh(range(len(top_features)), top_features[sort_column], color='skyblue')
plt.yticks(range(len(top_features)), top_features.index)
plt.xlabel('Важность признака (%)')
plt.title(f'ТОП-15 важных признаков ({sort_column})')
plt.gca().invert_yaxis()
plt.grid(True, alpha=0.3)

# График 2: Кумулятивная важность признаков
plt.subplot(2, 2, 2)
cumulative_importance = np.cumsum(feature_importance_df[sort_column])
plt.plot(range(1, len(cumulative_importance) + 1), cumulative_importance, marker='o', linewidth=2)
plt.axhline(y=95, color='red', linestyle='--', alpha=0.7, label='95% важности')
plt.axhline(y=90, color='orange', linestyle='--', alpha=0.7, label='90% важности')
plt.axhline(y=80, color='green', linestyle='--', alpha=0.7, label='80% важности')
plt.xlabel('Количество признаков')
plt.ylabel('Кумулятивная важность (%)')
plt.title(f'Кумулятивная важность признаков ({sort_column})')
plt.legend()
plt.grid(True, alpha=0.3)

# График 3: Сравнение моделей (ТОП-10 признаков)
plt.subplot(2, 2, 3)
top_10_features = feature_importance_df.head(10)
x = np.arange(len(top_10_features))
width = 0.25

available_models = [col for col in ['RandomForest', 'XGBoost', 'LightGBM'] if col in feature_importance_df.columns]

if len(available_models) >= 2:
    colors = ['purple', 'orange', 'green']
    for i, model in enumerate(available_models):
        offset = width * (i - (len(available_models)-1)/2)
        plt.bar(x + offset, top_10_features[model], width, label=model, alpha=0.8, color=colors[i])
    
    plt.xlabel('Признаки')
    plt.ylabel('Важность (%)')
    plt.title('Сравнение важности признаков между моделями')
    plt.xticks(x, top_10_features.index, rotation=45, ha='right')
    plt.legend()
    plt.grid(True, alpha=0.3)
else:
    plt.text(0.5, 0.5, 'Недостаточно данных\nдля сравнения моделей', 
            ha='center', va='center', transform=plt.gca().transAxes)
    plt.title('Сравнение важности признаков')

# График 4: УЛУЧШЕННАЯ КАТЕГОРИЗАЦИЯ (БЕЗ "ДРУГИХ")
plt.subplot(2, 2, 4)

# УМНАЯ КАТЕГОРИЗАЦИЯ - КАЖДЫЙ ПРИЗНАК ТОЛЬКО В ОДНОЙ КАТЕГОРИИ
feature_categories_smart = {}

# Сначала категории с четкими правилами (в порядке приоритета)
categories_priority = [
    ('Потребление по зонам', lambda f: any(x in f for x in [
        'sub_metering', 'kitchen', 'laundry', 'ac_heating', 'appliance'
    ])),
    ('Временные лаги', lambda f: any(x in f for x in ['lag_', 'momentum', 'deviation'])),
    ('Скользящие статистики', lambda f: 'rolling_' in f),
    ('Циклические', lambda f: 'sin' in f or 'cos' in f),
    ('Сезонность', lambda f: any(x in f for x in [
        'season', 'winter', 'summer', 'spring', 'year', 'month_'
    ]) and 'sin' not in f and 'cos' not in f),  # исключаем циклические
    ('Пиковые периоды', lambda f: any(x in f for x in [
        'peak', 'surge', 'night', 'morning', 'evening'
    ]) and not f.startswith('is_')),  # исключаем простые флаги
    ('Временные паттерны', lambda f: f in ['hour', 'day_of_week', 'month', 'day_of_year']),
    ('Бинарные флаги', lambda f: f.startswith('is_')),
]

# Распределяем признаки по категориям
used_features = set()
for category_name, condition in categories_priority:
    category_features = [f for f in model_features if condition(f) and f not in used_features]
    if category_features:
        feature_categories_smart[category_name] = category_features
        used_features.update(category_features)

# Всё что осталось - это "Производные признаки"
remaining_features = [f for f in model_features if f not in used_features]
if remaining_features:
    feature_categories_smart['Производные признаки'] = remaining_features

# Считаем важность по категориям
category_importance = {}
for category, features in feature_categories_smart.items():
    if features:
        importance_sum = feature_importance_df.loc[features, sort_column].sum()
        category_importance[category] = importance_sum

# Выводим отладочную информацию
print(f"\n=== РАСПРЕДЕЛЕНИЕ ПРИЗНАКОВ ПО КАТЕГОРИЯМ ===")
for category, features in feature_categories_smart.items():
    print(f"{category}: {len(features)} признаков")
    for feature in features[:5]:  # показываем первые 5 признаков
        importance = feature_importance_df.loc[feature, sort_column]
        print(f"  - {feature} ({importance:.2f}%)")
    if len(features) > 5:
        print(f"  ... и еще {len(features) - 5} признаков")
    print()

# ГОРИЗОНТАЛЬНАЯ ДИАГРАММА (лучшая читаемость)
if category_importance:
    sorted_categories = dict(sorted(category_importance.items(), key=lambda x: x[1], reverse=True))
    
    # Горизонтальная бар-диаграмма
    categories = list(sorted_categories.keys())
    importances = list(sorted_categories.values())
    
    bars = plt.barh(range(len(categories)), importances, 
                   color=plt.cm.Set3(np.linspace(0, 1, len(categories))),
                   alpha=0.7)
    
    plt.yticks(range(len(categories)), categories, fontsize=10)
    plt.xlabel('Важность (%)', fontsize=11)
    plt.title('Распределение важности по категориям признаков', fontsize=12)
    
    # Добавляем значения на барчики
    for i, (bar, importance) in enumerate(zip(bars, importances)):
        plt.text(bar.get_width() + 0.5, bar.get_y() + bar.get_height()/2, 
                f'{importance:.1f}%', 
                ha='left', va='center', fontsize=9)
    
    plt.grid(True, alpha=0.3, axis='x')
    plt.gca().invert_yaxis()  # Чтобы самая важная категория была сверху

plt.tight_layout()
plt.show()

# Анализ критических признаков
print("\n" + "=" * 60)
print("АНАЛИЗ КРИТИЧЕСКИХ ПРИЗНАКОВ")
print("=" * 60)

n_features_80 = np.argmax(cumulative_importance >= 80) + 1
n_features_90 = np.argmax(cumulative_importance >= 90) + 1  
n_features_95 = np.argmax(cumulative_importance >= 95) + 1

print(f"Признаков для 80% важности: {n_features_80}")
print(f"Признаков для 90% важности: {n_features_90}")
print(f"Признаков для 95% важности: {n_features_95}")

print(f"\nСамые важные признаки (топ-{n_features_80}):")
critical_features = feature_importance_df.head(n_features_80).index.tolist()
for i, feature in enumerate(critical_features, 1):
    importance = feature_importance_df.loc[feature, sort_column]
    category = next((cat for cat, features in feature_categories_smart.items() if feature in features), 'Не определено')
    print(f"  {i:2d}. {feature:<25} {importance:.2f}% ({category})")

print(f"\nРАСПРЕДЕЛЕНИЕ ПО КАТЕГОРИЯМ:")
for category, importance in sorted(category_importance.items(), key=lambda x: x[1], reverse=True):
    print(f"  {category}: {importance:.1f}%")


# Дополнительный анализ в конце
print(f"\n=== ЧЕСТНЫЙ АНАЛИЗ РЕЗУЛЬТАТОВ ===")
honest_categories = ['Временные лаги', 'Скользящие статистики', 'Циклические', 
                    'Временные паттерны', 'Бинарные флаги']
leakage_categories = ['Потребление по зонам']

honest_importance = sum(category_importance.get(cat, 0) for cat in honest_categories)
leakage_importance = sum(category_importance.get(cat, 0) for cat in leakage_categories)

print(f"Честные признаки (доступны в реальном времени): {honest_importance:.1f}%")
print(f"Признаки с потенциальной утечкой: {leakage_importance:.1f}%")
print(f"Прочие признаки: {100 - honest_importance - leakage_importance:.1f}%")

if honest_importance > leakage_importance:
    print("✅ Преобладают честные признаки - модель имеет потенциал для реального применения")
else:
    print("⚠️  Высокая зависимость от признаков с утечками - требуется доработка")

print(f"\nИНТЕРПРЕТАЦИЯ ЗАВЕРШЕНА!")
============================================================
ИНТЕРПРЕТАЦИЯ МОДЕЛЕЙ - АНАЛИЗ ВАЖНОСТИ ПРИЗНАКОВ
============================================================
Модели обучены на 33 признаках

=== ПРОВЕРКА КОРРЕКТНОСТИ ДАННЫХ ===
RandomForest: сумма важностей = 100.0%
XGBoost: сумма важностей = 100.0%
LightGBM: сумма важностей = 100.0%

ТОП-15 самых важных признаков по версии LightGBM:
==================================================
 1. lag_1h                    13.23%
 2. momentum_1h               11.87%
 3. acceleration_1h           9.13%
 4. rolling_std_6h            6.57%
 5. rolling_mean_6h           5.80%
 6. rolling_mean_720h         5.20%
 7. rolling_std_24h           5.07%
 8. volatility_ratio_6h       4.13%
 9. rolling_mean_24h          4.13%
10. long_term_seasonal_safe   3.17%
11. surge_6_7_safe            3.07%
12. deviation_from_daily_norm 2.90%
13. composite_energy_score    2.90%
14. lag_24h                   2.67%
15. seasonal_volatility       2.47%

=== РАСПРЕДЕЛЕНИЕ ПРИЗНАКОВ ПО КАТЕГОРИЯМ ===
Временные лаги: 9 признаков
  - lag_1h (13.23%)
  - lag_24h (2.67%)
  - lag_168h (1.23%)
  - momentum_1h (11.87%)
  - deviation_from_daily_norm (2.90%)
  ... и еще 4 признаков

Скользящие статистики: 5 признаков
  - rolling_mean_6h (5.80%)
  - rolling_mean_24h (4.13%)
  - rolling_mean_720h (5.20%)
  - rolling_std_6h (6.57%)
  - rolling_std_24h (5.07%)

Циклические: 5 признаков
  - hour_sin (2.33%)
  - hour_cos (1.37%)
  - month_sin (0.20%)
  - day_of_year_sin (2.20%)
  - day_of_year_cos (0.13%)

Сезонность: 3 признаков
  - spring_transition_effect (0.43%)
  - seasonal_volatility (2.47%)
  - long_term_seasonal_safe (3.17%)

Пиковые периоды: 5 признаков
  - morning_surge_6_7 (0.10%)
  - surge_6_7_safe (3.07%)
  - evening_intensity_safe (1.00%)
  - evening_hours (0.00%)
  - morning_hours (0.10%)

Бинарные флаги: 2 признаков
  - is_weekend (0.07%)
  - is_family_time (0.00%)

Производные признаки: 4 признаков
  - acceleration_1h (9.13%)
  - volatility_ratio_6h (4.13%)
  - weather_behavior_safe (0.93%)
  - composite_energy_score (2.90%)


============================================================
АНАЛИЗ КРИТИЧЕСКИХ ПРИЗНАКОВ
============================================================
Признаков для 80% важности: 15
Признаков для 90% важности: 19
Признаков для 95% важности: 22

Самые важные признаки (топ-15):
   1. lag_1h                    13.23% (Временные лаги)
   2. momentum_1h               11.87% (Временные лаги)
   3. acceleration_1h           9.13% (Производные признаки)
   4. rolling_std_6h            6.57% (Скользящие статистики)
   5. rolling_mean_6h           5.80% (Скользящие статистики)
   6. rolling_mean_720h         5.20% (Скользящие статистики)
   7. rolling_std_24h           5.07% (Скользящие статистики)
   8. volatility_ratio_6h       4.13% (Производные признаки)
   9. rolling_mean_24h          4.13% (Скользящие статистики)
  10. long_term_seasonal_safe   3.17% (Сезонность)
  11. surge_6_7_safe            3.07% (Пиковые периоды)
  12. deviation_from_daily_norm 2.90% (Временные лаги)
  13. composite_energy_score    2.90% (Производные признаки)
  14. lag_24h                   2.67% (Временные лаги)
  15. seasonal_volatility       2.47% (Сезонность)

РАСПРЕДЕЛЕНИЕ ПО КАТЕГОРИЯМ:
  Временные лаги: 39.5%
  Скользящие статистики: 26.8%
  Производные признаки: 17.1%
  Циклические: 6.2%
  Сезонность: 6.1%
  Пиковые периоды: 4.3%
  Бинарные флаги: 0.1%

=== ЧЕСТНЫЙ АНАЛИЗ РЕЗУЛЬТАТОВ ===
Честные признаки (доступны в реальном времени): 72.6%
Признаки с потенциальной утечкой: 0.0%
Прочие признаки: 27.4%
✅ Преобладают честные признаки - модель имеет потенциал для реального применения

ИНТЕРПРЕТАЦИЯ ЗАВЕРШЕНА!
# КОМБИНИРОВАННАЯ ВИЗУАЛИЗАЦИЯ - основная диаграмма + детализация
fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(16, 7))

if category_importance:
    sorted_categories = dict(sorted(category_importance.items(), key=lambda x: x[1], reverse=True))
    
    # ЛЕВАЯ ЧАСТЬ: Основная круговая диаграмма (топ-6 категорий)
    top_categories = dict(list(sorted_categories.items())[:5])
    other_total = sum(list(sorted_categories.values())[5:])
    
    if other_total > 0:
        top_categories['Остальные'] = other_total
    
    wedges, texts, autotexts = ax1.pie(
        top_categories.values(), 
        labels=top_categories.keys(), 
        autopct='%1.1f%%', 
        startangle=90,
        colors=plt.cm.Set3(np.linspace(0, 1, len(top_categories))),
        textprops={'fontsize': 10,'color': 'black'},
        pctdistance=0.8
    )
    
    for autotext in autotexts:
        autotext.set_color('black')
    
    ax1.set_title('Основные категории признаков\n(топ-6)', fontsize=12, fontweight='bold')
    
    # ПРАВАЯ ЧАСТЬ: Детализация маленьких категорий
    if len(sorted_categories) > 5:
        small_categories = dict(list(sorted_categories.items())[5:])
        categories = list(small_categories.keys())
        values = list(small_categories.values())
        
        bars = ax2.barh(range(len(categories)), values, 
                       color=plt.cm.Set3(np.linspace(0, 1, len(categories))),
                       alpha=0.7)
        
        ax2.set_yticks(range(len(categories)))
        ax2.set_yticklabels(categories, fontsize=9)
        ax2.set_xlabel('Важность (%)')
        ax2.set_title('Детализация малых категорий', fontsize=12, fontweight='bold')
        ax2.grid(True, alpha=0.3, axis='x')
        ax2.invert_yaxis()
        
        # Добавляем значения на барчики
        for i, (bar, value) in enumerate(zip(bars, values)):
            ax2.text(bar.get_width() + 0.1, bar.get_y() + bar.get_height()/2, 
                    f'{value:.1f}%', ha='left', va='center', fontsize=8)

plt.tight_layout()
plt.show()

# АНАЛИЗ ОШИБОК ПРОГНОЗИРОВАНИЯ
print("\n" + "=" * 60)
print("АНАЛИЗ ОШИБОК ПРОГНОЗИРОВАНИЯ")
print("=" * 60)

# Берем прогнозы лучшей модели
y_pred_best = results[f'{best_model}']['predictions']
errors = y_pred_best - y_test.values

# Создаем Series с ошибками и правильными индексами
errors_series = pd.Series(errors, index=y_test.index)

# Анализ больших ошибок
error_threshold = 0.05  # 50 Вт - порог для "большой" ошибки
large_errors_mask = np.abs(errors) > error_threshold
large_errors_count = np.sum(large_errors_mask)
large_errors_percentage = (large_errors_count / len(errors)) * 100

print(f"Большие ошибки (> {error_threshold} кВт): {large_errors_count} ({large_errors_percentage:.2f}%)")

# Для анализа временных характеристик создаем DataFrame с временными признаками из индекса
time_features_df = pd.DataFrame(index=X_test.index)
time_features_df['hour'] = time_features_df.index.hour
time_features_df['day_of_week'] = time_features_df.index.dayofweek
time_features_df['is_evening_peak'] = ((time_features_df['hour'] >= 18) & (time_features_df['hour'] <= 22)).astype(int)
time_features_df['is_morning_peak'] = ((time_features_df['hour'] >= 7) & (time_features_df['hour'] <= 9)).astype(int)
time_features_df['is_night'] = ((time_features_df['hour'] >= 0) & (time_features_df['hour'] <= 5)).astype(int)

if large_errors_count > 0:
    # Анализ когда происходят большие ошибки
    large_errors_time_data = time_features_df[large_errors_mask]
    
    print("\nХарактеристики периодов с большими ошибками:")
    print("Час дня:")
    print(large_errors_time_data['hour'].value_counts().sort_index())
    
    print("\nДень недели:")
    day_names = {0: 'Пн', 1: 'Вт', 2: 'Ср', 3: 'Чт', 4: 'Пт', 5: 'Сб', 6: 'Вс'}
    day_counts = large_errors_time_data['day_of_week'].value_counts().sort_index()
    for day, count in day_counts.items():
        print(f"  {day_names[day]}: {count} ошибок")
    
    print(f"\nПиковые периоды:")
    print(f"  Вечерний пик: {large_errors_time_data['is_evening_peak'].sum()} ошибок")
    print(f"  Утренний пик: {large_errors_time_data['is_morning_peak'].sum()} ошибок")
    print(f"  Ночное время: {large_errors_time_data['is_night'].sum()} ошибок")

# ВИЗУАЛИЗАЦИЯ ОШИБОК
plt.figure(figsize=(15, 5))

# График 1: Распределение ошибок по времени суток
plt.subplot(1, 3, 1)
hourly_errors = time_features_df.groupby('hour').apply(
    lambda x: np.mean(np.abs(errors_series[x.index])), include_groups=False
)
plt.bar(hourly_errors.index, hourly_errors.values, alpha=0.7, color='red')
plt.xlabel('Час дня')
plt.ylabel('Средняя абсолютная ошибка (кВт)')
plt.title('Ошибки по часам суток')
plt.grid(True, alpha=0.3)

# График 2: Ошибки по дням недели  
plt.subplot(1, 3, 2)
daily_errors = time_features_df.groupby('day_of_week').apply(
    lambda x: np.mean(np.abs(errors_series[x.index])), include_groups=False
)
plt.bar(daily_errors.index, daily_errors.values, alpha=0.7, color='orange')
plt.xlabel('День недели')
plt.ylabel('Средняя абсолютная ошибка (кВт)')
plt.title('Ошибки по дням недели')
plt.xticks(range(7), ['Пн', 'Вт', 'Ср', 'Чт', 'Пт', 'Сб', 'Вс'])
plt.grid(True, alpha=0.3)

# График 3: Факт vs Прогноз для худших случаев
plt.subplot(1, 3, 3)
worst_indices = np.argsort(np.abs(errors))[-10:]  # 10 худших прогнозов
worst_times = y_test.iloc[worst_indices].index

plt.plot(worst_times, y_test.iloc[worst_indices].values, 'o-', label='Факт', linewidth=2)
plt.plot(worst_times, y_pred_best[worst_indices], 'x-', label='Прогноз', linewidth=2)
plt.xlabel('Время')
plt.ylabel('Нагрузка (кВт)')
plt.title('10 худших прогнозов')
plt.legend()
plt.xticks(rotation=45)
plt.grid(True, alpha=0.3)

plt.tight_layout()
plt.show()

# ДОПОЛНИТЕЛЬНЫЙ АНАЛИЗ: Когда происходят самые большие ошибки?
print(f"\n=== АНАЛИЗ САМЫХ БОЛЬШИХ ОШИБОК ===")
top_5_worst_indices = np.argsort(np.abs(errors))[-5:]
top_5_worst_times = y_test.iloc[top_5_worst_indices].index
top_5_worst_errors = errors[top_5_worst_indices]

print("5 самых больших ошибок:")
for i, (time, error) in enumerate(zip(top_5_worst_times, top_5_worst_errors)):
    actual = y_test.iloc[top_5_worst_indices[i]]
    predicted = y_pred_best[top_5_worst_indices[i]]
    print(f"  {i+1}. {time}: факт={actual:.2f} кВт, прогноз={predicted:.2f} кВт, ошибка={error:.2f} кВт")

# Анализ характеристик в моменты самых больших ошибок
print(f"\nХарактеристики в моменты самых больших ошибок:")
for i, time in enumerate(top_5_worst_times):
    hour = time.hour
    day_of_week = time.dayofweek
    day_name = day_names[day_of_week]
    
    # Получаем значения признаков для этого времени (если есть в X_test)
    if time in X_test.index:
        time_data = X_test.loc[time]
        # Ищем самые экстремальные значения признаков
        extreme_features = time_data[np.abs(time_data) > time_data.abs().quantile(0.9)]
        print(f"  {i+1}. {time} ({day_name}, {hour:02d}:00): {len(extreme_features)} экстремальных признаков")

# Анализ сезонности ошибок
print(f"\n=== СЕЗОННЫЙ АНАЛИЗ ОШИБОК ===")
time_features_df['month'] = time_features_df.index.month
monthly_errors = time_features_df.groupby('month').apply(
    lambda x: np.mean(np.abs(errors_series[x.index])), include_groups=False
)

months_names = {1: 'Янв', 2: 'Фев', 3: 'Мар', 4: 'Апр', 5: 'Май', 6: 'Июн'}
print("Средние ошибки по месяцам:")
for month, error in monthly_errors.items():
    if month in months_names:
        print(f"  {months_names[month]}: {error:.3f} кВт")

# Анализ зависимости ошибок от величины потребления
print(f"\n=== АНАЛИЗ ЗАВИСИМОСТИ ОШИБОК ОТ ВЕЛИЧИНЫ ПОТРЕБЛЕНИЯ ===")
consumption_bins = [0, 0.5, 1.0, 2.0, 5.0, 10.0]
consumption_labels = ['Очень низкое', 'Низкое', 'Среднее', 'Высокое', 'Очень высокое']

y_test_binned = pd.cut(y_test, bins=consumption_bins, labels=consumption_labels)
error_by_consumption = pd.DataFrame({
    'consumption': y_test_binned,
    'abs_error': np.abs(errors)
}).groupby('consumption')['abs_error'].agg(['mean', 'std', 'count'])

print("Средние ошибки по уровням потребления:")
for level in consumption_labels:
    if level in error_by_consumption.index:
        data = error_by_consumption.loc[level]
        print(f"  {level}: {data['mean']:.3f} ± {data['std']:.3f} кВт (n={data['count']})")

print(f"\n✅ АНАЛИЗ ОШИБОК ЗАВЕРШЕН!")
============================================================
АНАЛИЗ ОШИБОК ПРОГНОЗИРОВАНИЯ
============================================================
Большие ошибки (> 0.05 кВт): 29747 (32.79%)

Характеристики периодов с большими ошибками:
Час дня:
hour
0     1008
1      836
2      913
3      820
4     1041
5      952
6     1350
7     1479
8     1160
9     1052
10    1159
11    1158
12    1091
13    1355
14    1358
15    1215
16    1122
17    1188
18    1382
19    1834
20    1718
21    1914
22    1570
23    1072
Name: count, dtype: int64

День недели:
  Пн: 3618 ошибок
  Вт: 4084 ошибок
  Ср: 4803 ошибок
  Чт: 3744 ошибок
  Пт: 4092 ошибок
  Сб: 4355 ошибок
  Вс: 5051 ошибок

Пиковые периоды:
  Вечерний пик: 8418 ошибок
  Утренний пик: 3691 ошибок
  Ночное время: 5570 ошибок

=== АНАЛИЗ САМЫХ БОЛЬШИХ ОШИБОК ===
5 самых больших ошибок:
  1. 2007-06-17 09:13:00: факт=3.70 кВт, прогноз=0.26 кВт, ошибка=-3.43 кВт
  2. 2007-05-06 08:07:00: факт=6.13 кВт, прогноз=2.69 кВт, ошибка=-3.44 кВт
  3. 2007-05-25 06:30:00: факт=4.60 кВт, прогноз=0.99 кВт, ошибка=-3.61 кВт
  4. 2007-06-28 07:02:00: факт=4.06 кВт, прогноз=0.44 кВт, ошибка=-3.62 кВт
  5. 2007-05-01 18:49:00: факт=4.89 кВт, прогноз=1.12 кВт, ошибка=-3.78 кВт

Характеристики в моменты самых больших ошибок:
  1. 2007-06-17 09:13:00 (Вс, 09:00): 4 экстремальных признаков
  2. 2007-05-06 08:07:00 (Вс, 08:00): 4 экстремальных признаков
  3. 2007-05-25 06:30:00 (Пт, 06:00): 4 экстремальных признаков
  4. 2007-06-28 07:02:00 (Чт, 07:00): 4 экстремальных признаков
  5. 2007-05-01 18:49:00 (Вт, 18:00): 4 экстремальных признаков

=== СЕЗОННЫЙ АНАЛИЗ ОШИБОК ===
Средние ошибки по месяцам:
  Апр: 0.037 кВт
  Май: 0.111 кВт
  Июн: 0.109 кВт

=== АНАЛИЗ ЗАВИСИМОСТИ ОШИБОК ОТ ВЕЛИЧИНЫ ПОТРЕБЛЕНИЯ ===
Средние ошибки по уровням потребления:
  Очень низкое: 0.042 ± 0.095 кВт (n=49249.0)
  Низкое: 0.094 ± 0.232 кВт (n=10049.0)
  Среднее: 0.112 ± 0.249 кВт (n=21926.0)
  Высокое: 0.431 ± 0.488 кВт (n=8958.0)
  Очень высокое: 0.825 ± 0.694 кВт (n=538.0)

✅ АНАЛИЗ ОШИБОК ЗАВЕРШЕН!
C:\Users\Andre\AppData\Local\Temp\ipykernel_1968\3676608647.py:139: FutureWarning: The default of observed=False is deprecated and will be changed to True in a future version of pandas. Pass observed=False to retain current behavior or observed=True to adopt the future default and silence this warning.
  }).groupby('consumption')['abs_error'].agg(['mean', 'std', 'count'])
# ===== СТРОГАЯ ПРОВЕРКА ВРЕМЕННОЙ СОГЛАСОВАННОСТИ =====
print("\n" + "="*60)
print("МАКСИМАЛЬНО СТРОГАЯ ПРОВЕРКА ВРЕМЕННЫХ ОКОН")
print("="*60)

# Тест 1: Проверка ВСЕХ признаков на временную согласованность
def analyze_feature_timing(feature_name, feature_series, target_series):
    """Анализирует временное окно признака"""
    # Находим индексы, где есть данные
    valid_idx = feature_series.first_valid_index()
    if valid_idx is None:
        return "❌ ВСЕ ЗНАЧЕНИЯ NaN"
    
    # Проверяем, можно ли вычислить признак в реальном времени
    required_data_points = []
    
    # Анализируем зависимость от целевой переменной
    if feature_name.startswith('lag_'):
        lag_period = int(feature_name.split('_')[1].replace('h', ''))
        required_data_points.append(f"t-{lag_period}")
    
    elif feature_name.startswith('rolling_'):
        # Для скользящих статистик - нужны данные до t-1
        required_data_points.append("t-1")
    
    elif feature_name in ['momentum_1h', 'acceleration_1h']:
        required_data_points.extend(["t-1", "t-2", "t-3"])
    
    elif feature_name in ['momentum_3h', 'momentum_24h']:
        required_data_points.extend(["t-3", "t-6", "t-24", "t-48"])
    
    return f"✅ Требует: {', '.join(required_data_points)}"

print("🔍 Анализ временных окон для ТОП-15 признаков:")
top_features = ['lag_1h', 'momentum_1h', 'acceleration_1h', 'rolling_std_6h', 
                'rolling_mean_6h', 'rolling_std_24h', 'volatility_ratio_6h',
                'rolling_mean_720h', 'surge_6_7_safe', 'rolling_mean_24h']

for feature in top_features:
    if feature in df.columns:
        timing_info = analyze_feature_timing(feature, df[feature], df['Global_active_power'])
        print(f"  {feature:<25} {timing_info}")

# Тест 2: Проверка на утечки в тестовой выборке
print(f"\n🔍 ПРОВЕРКА ТЕСТОВОЙ ВЫБОРКИ НА УТЕЧКИ:")

# Берем первую строку тестовой выборки
test_sample_time = X_test.index[0]
test_sample_data = X_test.iloc[0]

print(f"Время первого тестового прогноза: {test_sample_time}")

# Проверяем, какие данные используются для этого прогноза
lag_1h_time = test_sample_time - pd.Timedelta(hours=1)
lag_24h_time = test_sample_time - pd.Timedelta(hours=24)

print(f"Данные для lag_1h берутся из: {lag_1h_time}")
print(f"Данные для lag_24h берутся из: {lag_24h_time}")

# Проверяем, что эти временные точки ДЕЙСТВИТЕЛЬНО в обучающей выборке
if lag_1h_time in X_train.index:
    print("✅ lag_1h данные из обучающей выборки")
else:
    print("❌ УТЕЧКА: lag_1h данные НЕ из обучающей выборки!")

if lag_24h_time in X_train.index:
    print("✅ lag_24h данные из обучающей выборки") 
else:
    print("❌ УТЕЧКА: lag_24h данные НЕ из обучающей выборки!")

# Тест 3: Проверка временных разрывов
print(f"\n🔍 ПРОВЕРКА ВРЕМЕННЫХ РАЗРЫВОВ:")
train_end = X_train.index.max()
test_start = X_test.index.min()
gap = test_start - train_end

print(f"Обучающая выборка заканчивается: {train_end}")
print(f"Тестовая выборка начинается: {test_start}")
print(f"Разрыв: {gap}")

if gap > pd.Timedelta(minutes=10):
    print("❌ ПОДОЗРИТЕЛЬНО: Большой разрыв между train и test!")
else:
    print("✅ Временной разрыв в пределах нормы")

# Тест 4: Проверка распределения целевой переменной
print(f"\n🔍 ПРОВЕРКА РАСПРЕДЕЛЕНИЯ ЦЕЛЕВОЙ ПЕРЕМЕННОЙ:")
train_mean = y_train.mean()
test_mean = y_test.mean()
difference_pct = abs((test_mean - train_mean) / train_mean * 100)

print(f"Среднее train: {train_mean:.3f} кВт")
print(f"Среднее test:  {test_mean:.3f} кВт") 
print(f"Разница: {difference_pct:.1f}%")

if difference_pct > 20:
    print("⚠️  ЗНАЧИТЕЛЬНАЯ РАЗНИЦА в распределении!")
else:
    print("✅ Распределения схожи")

# Тест 5: Проверка на информационные утечки через даты
print(f"\n🔍 ПРОВЕРКА ИНФОРМАЦИОННЫХ УТЕЧЕК:")
train_dates = set(X_train.index.date)
test_dates = set(X_test.index.date)
overlap_dates = train_dates.intersection(test_dates)

if overlap_dates:
    print(f"❌ КРИТИЧЕСКАЯ УТЕЧКА: {len(overlap_dates)} общих дат между train и test!")
    print(f"   Общие даты: {sorted(overlap_dates)[:5]}...")
else:
    print("✅ Нет общих дат между train и test")

# Тест 6: Проверка корреляции признаков с "будущим"
print(f"\n🔍 ПРОВЕРКА КОРРЕЛЯЦИИ С БУДУЩИМИ ЗНАЧЕНИЯМИ:")

# Создаем "будущие" значения (сдвинутые на 1 период вперед)
y_future = df['Global_active_power'].shift(-1)

# Проверяем корреляцию только для обучающей выборки
train_corr_with_future = []
for feature in X_train.columns[:10]:  # Проверяем первые 10 признаков
    corr = X_train[feature].corr(y_future.loc[X_train.index])
    train_corr_with_future.append((feature, corr))

# Сортируем по абсолютной корреляции
train_corr_with_future.sort(key=lambda x: abs(x[1]), reverse=True)

print("Корреляция признаков с ЗНАЧЕНИЯМИ ИЗ БУДУЩЕГО:")
for feature, corr in train_corr_with_future[:5]:
    if abs(corr) > 0.3:
        print(f"  ❌ ПОДОЗРИТЕЛЬНО: {feature}: {corr:.3f}")
    else:
        print(f"  ✅ {feature}: {corr:.3f}")


# ДОБАВЬ ЭТОТ ТЕСТ:
print("\n🔍 ГЛУБОКИЙ АНАЛИЗ is_evening_peak:")
print(f"Уникальные значения: {df['is_evening_peak'].unique()}")
print(f"Распределение: {df['is_evening_peak'].value_counts()}")

# Проверим, нет ли утечки через этот признак
evening_peak_corr_with_future = df['is_evening_peak'].corr(df['Global_active_power'].shift(-1))
print(f"Корреляция is_evening_peak с будущим: {evening_peak_corr_with_future:.3f}")

if abs(evening_peak_corr_with_future) > 0.3:
    print("❌ ВОЗМОЖНА УТЕЧКА: is_evening_peak слишком связан с будущим!")
    # Рекомендуется пересчитать этот признак


# ТЕСТ НА СТРОГОЕ ВРЕМЕННОЕ РАЗДЕЛЕНИЕ
def strict_temporal_split_test():
    train_dates = set(X_train.index.date)
    test_dates = set(X_test.index.date)
    overlap = train_dates.intersection(test_dates)
    
    if overlap:
        print(f"❌ НЕИСПРАВЛЕНА УТЕЧКА: {len(overlap)} общих дат")
        return False
    else:
        print("✅ Утечка через даты исправлена")
        return True

# Проверить, что между train и test есть временной буфер
time_gap = X_test.index.min() - X_train.index.max()
if time_gap < pd.Timedelta(days=1):
    print("⚠️  СЛИШКОМ МАЛЕНЬКИЙ РАЗРЫВ МЕЖДУ TRAIN И TEST")



print("\n" + "="*60)
print("СТРОГАЯ ПРОВЕРКА ЗАВЕРШЕНА")
print("="*60)
============================================================
МАКСИМАЛЬНО СТРОГАЯ ПРОВЕРКА ВРЕМЕННЫХ ОКОН
============================================================
🔍 Анализ временных окон для ТОП-15 признаков:
  lag_1h                    ✅ Требует: t-1
  momentum_1h               ✅ Требует: t-1, t-2, t-3
  acceleration_1h           ✅ Требует: t-1, t-2, t-3
  rolling_std_6h            ✅ Требует: t-1
  rolling_mean_6h           ✅ Требует: t-1
  rolling_std_24h           ✅ Требует: t-1
  volatility_ratio_6h       ✅ Требует: 
  rolling_mean_720h         ✅ Требует: t-1
  surge_6_7_safe            ✅ Требует: 
  rolling_mean_24h          ✅ Требует: t-1

🔍 ПРОВЕРКА ТЕСТОВОЙ ВЫБОРКИ НА УТЕЧКИ:
Время первого тестового прогноза: 2007-04-29 00:00:00
Данные для lag_1h берутся из: 2007-04-28 23:00:00
Данные для lag_24h берутся из: 2007-04-28 00:00:00
❌ УТЕЧКА: lag_1h данные НЕ из обучающей выборки!
❌ УТЕЧКА: lag_24h данные НЕ из обучающей выборки!

🔍 ПРОВЕРКА ВРЕМЕННЫХ РАЗРЫВОВ:
Обучающая выборка заканчивается: 2007-04-14 23:59:00
Тестовая выборка начинается: 2007-04-29 00:00:00
Разрыв: 14 days 00:01:00
❌ ПОДОЗРИТЕЛЬНО: Большой разрыв между train и test!

🔍 ПРОВЕРКА РАСПРЕДЕЛЕНИЯ ЦЕЛЕВОЙ ПЕРЕМЕННОЙ:
Среднее train: 1.360 кВт
Среднее test:  0.898 кВт
Разница: 34.0%
⚠️  ЗНАЧИТЕЛЬНАЯ РАЗНИЦА в распределении!

🔍 ПРОВЕРКА ИНФОРМАЦИОННЫХ УТЕЧЕК:
✅ Нет общих дат между train и test

🔍 ПРОВЕРКА КОРРЕЛЯЦИИ С БУДУЩИМИ ЗНАЧЕНИЯМИ:
Корреляция признаков с ЗНАЧЕНИЯМИ ИЗ БУДУЩЕГО:
  ❌ ПОДОЗРИТЕЛЬНО: lag_1h: 0.932
  ❌ ПОДОЗРИТЕЛЬНО: lag_24h: 0.707
  ✅ is_family_time: 0.220
  ✅ hour_sin: -0.210
  ✅ is_weekend: 0.184

🔍 ГЛУБОКИЙ АНАЛИЗ is_evening_peak:
Уникальные значения: [0. 1.]
Распределение: is_evening_peak
0.0    206340
1.0     54300
Name: count, dtype: int64
Корреляция is_evening_peak с будущим: 0.323
❌ ВОЗМОЖНА УТЕЧКА: is_evening_peak слишком связан с будущим!

============================================================
СТРОГАЯ ПРОВЕРКА ЗАВЕРШЕНА
============================================================
# ===== ТЕСТ НА РЕАЛЬНОЕ ПРИМЕНЕНИЕ =====
print("\n🎯 ТЕСТ РЕАЛЬНОГО ПРИМЕНЕНИЯ В МОМЕНТ ВРЕМЕНИ t")

# Выбираем произвольный момент времени из тестовой выборки
test_time = X_test.index[100]
print(f"Тестируем момент времени: {test_time}")

# Какие данные ДОЛЖНЫ быть доступны в этот момент:
print("\nВ момент времени t ДОСТУПНЫ:")
print(f"  - Данные до {test_time - pd.Timedelta(minutes=1)}")
print(f"  - Временные признаки для {test_time}")

# Проверяем, можем ли мы вычислить ВСЕ признаки для этого времени
available_at_t = []
not_available_at_t = []

for feature in X_train.columns:
    if feature.startswith('lag_'):
        # Лаги требуют исторических данных
        available_at_t.append(feature)
    elif feature.startswith('rolling_'):
        # Скользящие статистики требуют исторических данных  
        available_at_t.append(feature)
    elif any(x in feature for x in ['hour', 'month', 'day', 'is_']):
        # Временные признаки - доступны всегда
        available_at_t.append(feature)
    else:
        # Все остальные - проверяем особо
        available_at_t.append(feature)

print(f"\n✅ Признаков, доступных в реальном времени: {len(available_at_t)}/{len(X_train.columns)}")
print(f"❌ Признаков, НЕ доступных в реальном времени: {len(not_available_at_t)}")

if not_available_at_t:
    print("КРИТИЧЕСКИЕ ПРОБЛЕМЫ:")
    for feature in not_available_at_t:
        print(f"  - {feature}")
else:
    print("🎉 ВСЕ признаки могут быть вычислены в реальном времени!")
🎯 ТЕСТ РЕАЛЬНОГО ПРИМЕНЕНИЯ В МОМЕНТ ВРЕМЕНИ t
Тестируем момент времени: 2007-04-29 01:40:00

В момент времени t ДОСТУПНЫ:
  - Данные до 2007-04-29 01:39:00
  - Временные признаки для 2007-04-29 01:40:00

✅ Признаков, доступных в реальном времени: 33/33
❌ Признаков, НЕ доступных в реальном времени: 0
🎉 ВСЕ признаки могут быть вычислены в реальном времени!
# СТРОГАЯ ПРОВЕРКА ЦЕЛОСТНОСТИ ПРИЗНАКОВ
print("\n=== СТРОГАЯ ПРОВЕРКА ЦЕЛОСТНОСТИ ===")

# 1. Проверяем, что все признаки существуют в df
missing_features = [f for f in available_features if f not in df.columns]
if missing_features:
    print(f"❌ ОТСУТСТВУЮЩИЕ ПРИЗНАКИ: {missing_features}")
else:
    print("✅ Все признаки присутствуют в данных")

# 2. Проверяем, что нет NaN-признаков
nan_features = []
for feature in available_features:
    if df[feature].isnull().all():  # Если ВСЕ значения NaN
        nan_features.append(feature)

if nan_features:
    print(f"❌ ПУСТЫЕ ПРИЗНАКИ (все значения NaN): {nan_features}")
else:
    print("✅ Нет пустых признаков")

# 3. Проверяем размерности
print(f"Размерность X_train: {X_train.shape}")
print(f"Количество признаков: {len(available_features)}")

# Должны совпадать!
if X_train.shape[1] == len(available_features):
    print("✅ Размерности согласованы")
else:
    print(f"❌ НЕСОГЛАСОВАННОСТЬ: X_train имеет {X_train.shape[1]} признаков, но в списке {len(available_features)}")
=== СТРОГАЯ ПРОВЕРКА ЦЕЛОСТНОСТИ ===
✅ Все признаки присутствуют в данных
✅ Нет пустых признаков
Размерность X_train: (149760, 33)
Количество признаков: 33
✅ Размерности согласованы
# РАЗДЕЛЕНИЕ ПО ВРЕМЕНИ (важно для временных рядов!)
split_date = '2007-05-01'  # Последние 2 месяца для теста
train = df[df.index < split_date]
test = df[df.index >= split_date]

# ПЕРЕОБУЧЕНИЕ НА ТЕСТЕ
lgb_model_retrained = LGBMRegressor()
lgb_model_retrained.fit(train[available_features], train['Global_active_power'])

test_predictions = lgb_model_retrained.predict(test[available_features])
test_mae = mean_absolute_error(test['Global_active_power'], test_predictions)

print(f"MAE на новых данных: {test_mae:.4f} кВт")
if test_mae > results['LightGBM']['MAE'] * 1.2:
    print("⚠️  ВОЗМОЖНОЕ ПЕРЕОБУЧЕНИЕ!")
MAE на новых данных: 0.1037 кВт
# СОХРАНЕНИЕ ТОЛЬКО ЛУЧШЕЙ МОДЕЛИ
print("Сохранение лучшей модели...")

# Сохраняем только лучшую модель
best_filename = f'models/{best_model.lower()}_best_model.pkl'
joblib.dump(models[best_model], best_filename)
print(f"Лучшая модель {best_model} сохранена в {best_filename}")

# Сохраняем метрики всех моделей для истории
results_df = pd.DataFrame(results).T
results_df.to_csv('models/model_metrics.csv')
print("Метрики всех моделей сохранены для сравнения")

print(f"\nВСЁ ЗАВЕРШЕНО! Лучшая модель: {best_model} (MAE: {results[f'{best_model}']['MAE']:.4f} кВт)")
Сохранение лучшей модели...
Лучшая модель LightGBM сохранена в models/lightgbm_best_model.pkl
Метрики всех моделей сохранены для сравнения

ВСЁ ЗАВЕРШЕНО! Лучшая модель: LightGBM (MAE: 0.1077 кВт)
# Сохраняем имена признаков
feature_names = X_train.columns.tolist()
print(f"Сохраняем {len(feature_names)} признаков")

# Сохраняем в файл
import json
with open('models/feature_names.json', 'w', encoding='utf-8') as f:
    json.dump(feature_names, f, ensure_ascii=False, indent=2)

print("Имена признаков сохранены в models/feature_names.json")
Сохраняем 33 признаков
Имена признаков сохранены в models/feature_names.json



Тебе НЕ НУЖНО писать код с новыми моделями, я уже его чутка сделал и отправлю тебе его следующим сообщением для проверки

XXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXX XXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXX XXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXX XXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXX XXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXX XXXXXXXXXXXXXXXXXXXXXX XXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXX XXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXX XXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXX XXXXXXXXXXXXXXXX XXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXX XXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXX XXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXX XXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXX XXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXX

Так теперь обучение новых моделей == 


import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
from sklearn.metrics import mean_absolute_error, mean_squared_error, r2_score
import warnings
warnings.filterwarnings('ignore')

# ДОБАВЬТЕ ЭТИ ИМПОРТЫ В НАЧАЛО КОДА:
from tensorflow.keras.callbacks import EarlyStopping
import xgboost as xgb
from catboost import CatBoostRegressor
from lightgbm import LGBMRegressor

print("📊 Начинаем переобучение с правильными моделями временных рядов...")
📊 Начинаем переобучение с правильными моделями временных рядов...
# Загрузка данных
df = pd.read_csv('df/obr.csv', parse_dates=['datetime'], index_col='datetime')
print(f"📈 Загружено данных: {df.shape}")

# Используем ваши лучшие признаки
best_features = [
    'lag_1h', 'momentum_1h', 'acceleration_1h', 'rolling_std_6h', 
    'rolling_mean_6h', 'rolling_mean_720h', 'rolling_std_24h', 
    'volatility_ratio_6h', 'rolling_mean_24h', 'long_term_seasonal_safe',
    'surge_6_7_safe', 'deviation_from_daily_norm', 'composite_energy_score',
    'lag_24h', 'seasonal_volatility', 'hour_sin', 'hour_cos', 
    'day_of_year_sin', 'day_of_year_cos', 'is_weekend'
]

# Целевая переменная
target = 'Global_active_power'
📈 Загружено данных: (260640, 14)
# УЛУЧШЕННОЕ СОЗДАНИЕ ПРИЗНАКОВ ПОЛНОСТЬЮ БЕЗ УТЕЧЕК
print("Создание УЛУЧШЕННЫХ признаков ПОЛНОСТЬЮ БЕЗ УТЕЧЕК...")
df = pd.read_csv('df/obr.csv', parse_dates=['datetime'], index_col='datetime')
print(f"Исходный размер: {df.shape}")

# ===== ОСНОВНЫЕ ВРЕМЕННЫЕ ПРИЗНАКИ (БЕЗОПАСНЫЕ) =====
# Циклические признаки - ✅ БЕЗОПАСНО
df['hour_sin'] = np.sin(2 * np.pi * df['hour']/24)
df['hour_cos'] = np.cos(2 * np.pi * df['hour']/24)
df['month_sin'] = np.sin(2 * np.pi * df['month']/12)
df['month_cos'] = np.cos(2 * np.pi * df['month']/12)
df['day_of_week_sin'] = np.sin(2 * np.pi * df['day_of_week']/7)
df['day_of_week_cos'] = np.cos(2 * np.pi * df['day_of_week']/7)
df['day_of_year'] = df.index.dayofyear
df['day_of_year_sin'] = np.sin(2 * np.pi * df['day_of_year']/365)
df['day_of_year_cos'] = np.cos(2 * np.pi * df['day_of_year']/365)

# ===== ВРЕМЕННЫЕ ПАТТЕРНЫ (БЕЗОПАСНЫЕ) =====
# СУТОЧНЫЕ ПАТТЕРНЫ
df['is_early_morning'] = ((df['hour'] >= 4) & (df['hour'] <= 6)).astype(int)
df['is_midday'] = ((df['hour'] >= 10) & (df['hour'] <= 16)).astype(int)
df['is_late_evening'] = ((df['hour'] >= 21) & (df['hour'] <= 23)).astype(int)
# df['is_evening_peak'] = ((df['hour'] >= 18) & (df['hour'] <= 22)).astype(int)
# df['is_morning_peak'] = ((df['hour'] >= 7) & (df['hour'] <= 9)).astype(int)
df['is_night'] = ((df['hour'] >= 0) & (df['hour'] <= 5)).astype(int)
df['is_deep_night'] = ((df['hour'] >= 1) & (df['hour'] <= 4)).astype(int)

# НЕДЕЛЬНЫЕ ПАТТЕРНЫ
df['is_monday'] = (df['day_of_week'] == 0).astype(int)
df['is_friday'] = (df['day_of_week'] == 4).astype(int)
df['is_sunday'] = (df['day_of_week'] == 6).astype(int)
df['is_week_start'] = (df['day_of_week'].isin([0, 1])).astype(int)
df['is_week_end'] = (df['day_of_week'].isin([4, 5])).astype(int)
df['is_family_time'] = ((df['hour'] >= 18) & (df['hour'] <= 22) & (df['is_weekend'] == 0)).astype(int)

# СЕЗОННЫЕ ПАТТЕРНЫ
df['is_high_season'] = df['month'].isin([1, 2, 12]).astype(int)
df['is_low_season'] = df['month'].isin([6, 7, 8]).astype(int)
df['is_spring'] = df['month'].isin([3, 4, 5]).astype(int)

# КРИТИЧЕСКИЕ ПЕРИОДЫ
df['morning_surge_6_7'] = ((df['hour'] >= 6) & (df['hour'] <= 7)).astype(int)
df['evening_surge_17_18'] = ((df['hour'] >= 17) & (df['hour'] <= 18)).astype(int)
df['evening_drop_22_23'] = ((df['hour'] >= 22) & (df['hour'] <= 23)).astype(int)

# ВЗАИМОДЕЙСТВИЯ ВРЕМЕНИ
df['winter_evening'] = (df['is_high_season'] & df['is_evening_peak']).astype(int)
df['summer_afternoon'] = (df['is_low_season'] & df['is_midday']).astype(int)
df['workday_evening'] = ((~df['is_weekend']) & df['is_evening_peak']).astype(int)
df['sunday_evening'] = (df['is_sunday'] & df['is_evening_peak']).astype(int)
df['weekend_evening_boost'] = (df['is_weekend'] & df['is_evening_peak']).astype(int)
df['weekend_morning'] = (df['is_weekend'] & df['is_morning_peak']).astype(int)
df['winter_weekend_evening'] = (df['is_high_season'] & df['is_weekend'] & df['is_evening_peak']).astype(int)

# ===== ЛАГИ ЦЕЛЕВОЙ ПЕРЕМЕННОЙ (БЕЗОПАСНЫЕ) =====
print("Создание лагов на основе анализа автокорреляции...")
df['lag_1h'] = df['Global_active_power'].shift(1)    # 1 час назад
df['lag_2h'] = df['Global_active_power'].shift(2)    # 2 часа назад
df['lag_3h'] = df['Global_active_power'].shift(3)    # 3 часа назад
df['lag_6h'] = df['Global_active_power'].shift(6)    # 6 часов назад
df['lag_12h'] = df['Global_active_power'].shift(12)  # 12 часов назад
df['lag_24h'] = df['Global_active_power'].shift(24)  # 1 день назад
df['lag_48h'] = df['Global_active_power'].shift(48)  # 2 дня назад  
df['lag_72h'] = df['Global_active_power'].shift(72)  # 3 дня назад
df['lag_168h'] = df['Global_active_power'].shift(168)  # 1 неделя назад

# ===== СКОЛЬЗЯЩИЕ СТАТИСТИКИ (БЕЗОПАСНЫЕ) =====
print("Создание скользящих статистик БЕЗ утечек...")
# Двойной сдвиг для полной безопасности
df['rolling_mean_6h'] = df['Global_active_power'].shift(1).rolling(6, min_periods=1).mean()
df['rolling_mean_12h'] = df['Global_active_power'].shift(1).rolling(12, min_periods=1).mean()
df['rolling_mean_24h'] = df['Global_active_power'].shift(1).rolling(24, min_periods=1).mean()
df['rolling_mean_72h'] = df['Global_active_power'].shift(1).rolling(72, min_periods=1).mean()
df['rolling_mean_168h'] = df['Global_active_power'].shift(1).rolling(168, min_periods=1).mean()
df['rolling_mean_720h'] = df['Global_active_power'].shift(1).rolling(720, min_periods=1).mean()

df['rolling_std_6h'] = df['Global_active_power'].shift(1).rolling(6, min_periods=1).std()
df['rolling_std_12h'] = df['Global_active_power'].shift(1).rolling(12, min_periods=1).std()
df['rolling_std_24h'] = df['Global_active_power'].shift(1).rolling(24, min_periods=1).std()
df['rolling_std_72h'] = df['Global_active_power'].shift(1).rolling(72, min_periods=1).std()
df['rolling_std_168h'] = df['Global_active_power'].shift(1).rolling(168, min_periods=1).std()

# ===== ПРОИЗВОДНЫЕ ОТ ЛАГОВ (БЕЗОПАСНЫЕ) =====
print("Создание производных от лагов...")
# df['momentum_1h'] = df['lag_1h'] - df['Global_active_power'].shift(2)  # Тренд за 1 час
df['momentum_1h'] = df['Global_active_power'].shift(1) - df['Global_active_power'].shift(2)
# df['momentum_3h'] = df['lag_3h'] - df['Global_active_power'].shift(6)  # Тренд за 3 часа
# df['momentum_3h'] = df['lag_3h'] - df['lag_6h']
# df['momentum_24h'] = df['lag_24h'] - df['Global_active_power'].shift(48)  # Тренд за сутки
# df['momentum_24h'] = df['lag_24h'] - df['lag_48h']

# df['acceleration_1h'] = df['momentum_1h'] - (df['lag_1h'] - df['Global_active_power'].shift(2)).shift(1)
df['acceleration_1h'] = (df['Global_active_power'].shift(1) - 2*df['Global_active_power'].shift(2) + df['Global_active_power'].shift(3))

df['deviation_from_daily_norm'] = df['lag_24h'] - df['rolling_mean_24h']
# df['deviation_from_weekly_norm'] = df['lag_168h'] - df['rolling_mean_168h']

df['volatility_ratio_6h'] = df['rolling_std_6h'] / (df['rolling_mean_6h'] + 0.001)
df['volatility_ratio_24h'] = df['rolling_std_24h'] / (df['rolling_mean_24h'] + 0.001)

# df['weekly_seasonal_factor'] = df['lag_168h'] / (df['rolling_mean_168h'] + 0.001)
# df['weekly_seasonal_factor'] = df['lag_168h'] / (df['rolling_mean_168h'].shift(167) + 0.001)
df['daily_seasonal_factor'] = df['lag_24h'] / (df['rolling_mean_24h'] + 0.001)

# ===== ФИЗИЧЕСКИЕ ПРИЗНАКИ (БЕЗОПАСНЫЕ) =====
# print("Создание физических признаков...")
# df['load_behavior_lag_1h'] = df['Global_intensity'].shift(1) * df['Voltage'].shift(1) / 1000
# df['power_factor_estimate'] = df['Global_active_power'].shift(1) / (df['load_behavior_lag_1h'] + 0.001)

# ===== ВЗАИМОДЕЙСТВИЯ ЛАГОВ С ВРЕМЕНЕМ (БЕЗОПАСНЫЕ) =====
print("Создание взаимодействий лагов...")
df['lag_1h_morning_boost'] = df['lag_1h'] * df['is_morning_peak']
df['lag_1h_evening_boost'] = df['lag_1h'] * df['is_evening_peak']
df['lag_24h_weekend_effect'] = df['lag_24h'] * df['is_weekend']
df['lag_168h_seasonal_boost'] = df['lag_168h'] * df['month_sin']

df['momentum_evening_intensity'] = df['momentum_1h'] * df['is_evening_peak']
df['volatility_morning_effect'] = df['rolling_std_24h'] * df['is_morning_peak']

# ===== СЕЗОННЫЕ ВЗАИМОДЕЙСТВИЯ (БЕЗОПАСНЫЕ) =====
print("Создание сезонных взаимодействий...")
df['winter_evening_demand'] = df['is_high_season'] * df['is_evening_peak'] * df['lag_24h']
df['summer_afternoon_cooling'] = df['is_low_season'] * df['is_midday'] * df['rolling_mean_24h']  # Безопасная замена
df['spring_transition_effect'] = df['is_spring'] * df['day_of_year_sin'] * df['rolling_std_24h']

df['seasonal_volatility'] = df['day_of_year_sin'] * df['rolling_std_24h']
df['weekly_seasonal_intensity'] = df['lag_168h'] * df['is_weekend'] * df['rolling_mean_24h']

# ===== ПОВЕДЕНЧЕСКИЕ ПАТТЕРНЫ (БЕЗОПАСНЫЕ) =====
print("Создание поведенческих паттернов...")
df['evening_peak_intensity_safe'] = df['is_evening_peak'] * df['rolling_std_24h'] * df['rolling_mean_24h']  # Безопасная замена
df['morning_peak_momentum_safe'] = df['is_morning_peak'] * df['momentum_1h']
df['weekend_lifestyle_safe'] = df['is_weekend'] * df['rolling_mean_24h'] * df['rolling_std_24h']  # Безопасная замена
df['workday_routine_safe'] = (~df['is_weekend'].astype(bool)).astype(int) * df['rolling_std_24h'] * df['rolling_mean_24h']  # Безопасная замена

# ===== ДОПОЛНИТЕЛЬНЫЕ БЕЗОПАСНЫЕ ПРИЗНАКИ =====
print("Создание дополнительных безопасных признаков...")
# Паттерны пиковых периодов
df['peak_morning_7_9'] = ((df['hour'] >= 7) & (df['hour'] <= 9)).astype(int)
df['peak_evening_18_22'] = ((df['hour'] >= 18) & (df['hour'] <= 22)).astype(int)

df['morning_intensity_safe'] = df['peak_morning_7_9'] * df['rolling_std_24h'] * df['lag_1h']
df['evening_intensity_safe'] = df['peak_evening_18_22'] * df['rolling_std_24h'] * df['lag_24h']

# Критические переходы
df['surge_6_7_safe'] = df['morning_surge_6_7'] * df['momentum_1h']
df['drop_22_23_safe'] = df['evening_drop_22_23'] * df['rolling_std_24h']

# Сложные сезонные взаимодействия
df['seasonal_energy_demand_safe'] = df['day_of_year_sin'] * df['month_sin'] * df['rolling_std_24h'] * df['rolling_mean_24h']  # Безопасная замена
df['weather_behavior_safe'] = df['day_of_year_cos'] * df['is_high_season'] * df['lag_24h']
df['monthly_energy_cycle_safe'] = df['month_sin'] * df['rolling_mean_168h'] * df['rolling_std_24h']  # Безопасная замена

df['seasonal_peak_intensity_safe'] = (
    df['is_evening_peak'] * 
    df['day_of_year_sin'] * 
    (df['is_high_season'] * 1.5 + df['is_low_season'] * 0.7)
)

df['long_term_seasonal_safe'] = df['day_of_year_sin'] * df['rolling_mean_720h']

# ===== ФИНАЛЬНЫЕ ПРОИЗВОДНЫЕ (БЕЗОПАСНЫЕ) =====
print("Создание финальных производных признаков...")
df['composite_energy_score'] = (
    df['lag_1h'] * 0.3 +
    df['momentum_1h'] * 0.2 + 
    df['rolling_mean_24h'] * 0.2 +
    df['rolling_std_24h'] * 0.15 +
    df['day_of_year_sin'] * 0.15
)

df['behavioral_energy_pattern'] = (
    df['is_evening_peak'] * df['lag_1h'] * 0.4 +
    df['is_weekend'] * df['rolling_mean_24h'] * 0.3 +
    df['is_high_season'] * df['rolling_std_24h'] * 0.3
)

# ===== ОБРАБОТКА ПРОПУСКОВ =====
print("Обработка пропусков...")
initial_size = len(df)

# Заполняем пропуски в числовых признаках медианой
numeric_columns = df.select_dtypes(include=[np.number]).columns
for col in numeric_columns:
    if df[col].isnull().any():
        df[col] = df[col].fillna(df[col].median())

# Удаляем оставшиеся пропуски (если есть)
df = df.dropna()

print(f"Признаки созданы. Удалено {initial_size - len(df)} строк с пропусками")
print(f"Финальный размер: {df.shape}")
print(f"Создано признаков: {len([col for col in df.columns if col not in ['Global_active_power', 'datetime']])}")

# ПРЕОБРАЗОВАНИЕ К FLOAT32
print("Преобразование признаков к float32...")
for col in df.columns:
    if col != 'Global_active_power':
        df[col] = df[col].astype('float32')

print("✅ Все признаки созданы ПОЛНОСТЬЮ БЕЗ УТЕЧЕК!")
Создание УЛУЧШЕННЫХ признаков ПОЛНОСТЬЮ БЕЗ УТЕЧЕК...
Исходный размер: (260640, 14)
Создание лагов на основе анализа автокорреляции...
Создание скользящих статистик БЕЗ утечек...
Создание производных от лагов...
Создание взаимодействий лагов...
Создание сезонных взаимодействий...
Создание поведенческих паттернов...
Создание дополнительных безопасных признаков...
Создание финальных производных признаков...
Обработка пропусков...
Признаки созданы. Удалено 0 строк с пропусками
Финальный размер: (260640, 100)
Создано признаков: 99
Преобразование признаков к float32...
✅ Все признаки созданы ПОЛНОСТЬЮ БЕЗ УТЕЧЕК!
# УДАЛИТЬ ВСЕ ПРИЗНАКИ, ИСПОЛЬЗУЮЩИЕ ПРОБЛЕМНЫЕ ПЕРЕМЕННЫЕ
leaky_features = [
    'winter_evening', 'workday_evening', 'sunday_evening', 
    'weekend_evening_boost', 'weekend_morning', 'winter_weekend_evening',
    'lag_1h_evening_boost', 'momentum_evening_intensity', 'volatility_morning_effect',
    'winter_evening_demand', 'evening_peak_intensity_safe', 'morning_peak_momentum_safe',
    'seasonal_peak_intensity_safe'
]

# Удаляем из DataFrame
df = df.drop(columns=[f for f in leaky_features if f in df.columns])
print(f"Удалено {len(leaky_features)} признаков с утечками")


# ЗАМЕНИТЬ проблемные признаки на безопасные аналоги
df['evening_hours'] = ((df['hour'] >= 18) & (df['hour'] <= 22)).astype(int)
df['morning_hours'] = ((df['hour'] >= 7) & (df['hour'] <= 9)).astype(int)

# Безопасные версии взаимодействий
df['lag_1h_evening_safe'] = df['lag_1h'] * df['evening_hours']
df['momentum_evening_safe'] = df['momentum_1h'] * df['evening_hours']
Удалено 13 признаков с утечками
print("\n⏰ Правильное временное разделение...")

# Последовательное разделение (временная валидация)
train_end = '2007-04-30'    # 4 месяца обучения
val_start = '2007-05-01'    # 1 месяц валидации  
val_end = '2007-05-31'
test_start = '2007-06-01'   # 1 месяц тестирования

# Создаем разделения
train_data = df[:train_end]
val_data = df[val_start:val_end]
test_data = df[test_start:]

print(f"Обучающая выборка: {train_data.shape[0]:,} записей ({train_data.index.min()} - {train_data.index.max()})")
print(f"Валидационная выборка: {val_data.shape[0]:,} записей ({val_data.index.min()} - {val_data.index.max()})")
print(f"Тестовая выборка: {test_data.shape[0]:,} записей ({test_data.index.min()} - {test_data.index.max()})")
⏰ Правильное временное разделение...
Обучающая выборка: 172,800 записей (2007-01-01 00:00:00 - 2007-04-30 23:59:00)
Валидационная выборка: 44,640 записей (2007-05-01 00:00:00 - 2007-05-31 23:59:00)
Тестовая выборка: 43,200 записей (2007-06-01 00:00:00 - 2007-06-30 23:59:00)
print("\n📊 Обучение SARIMA модели...")

from statsmodels.tsa.statespace.sarimax import SARIMAX
from statsmodels.tsa.stattools import adfuller

# Проверяем стационарность
def check_stationarity(series):
    result = adfuller(series.dropna())
    return result[1] < 0.05  # p-value < 0.05 = стационарный

# Подготовка данных для SARIMA (дневные агрегаты)
daily_data = df[target].resample('D').mean()

# Декомпозиция для определения параметров
from statsmodels.tsa.seasonal import seasonal_decompose
decomposition = seasonal_decompose(daily_data.dropna(), period=7, model='additive')

print("🎯 Определение параметров SARIMA:")
print(f"Сезонность: 7 дней (недельная)")
print(f"Тренд: {'Есть' if decomposition.trend is not None else 'Нет'}")
print(f"Стационарность: {'Да' if check_stationarity(daily_data) else 'Нет'}")

# SARIMA параметры на основе вашего EDA
sarima_order = (1, 1, 1)           # (p, d, q) - автокорреляция, дифференцирование, скользящее среднее
seasonal_order = (1, 1, 1, 24)     # (P, D, Q, s) - сезонность 24 часа

try:
    # Обучаем на тренировочных данных
    sarima_train = train_data[target].resample('H').mean()  # Часовые агрегаты для SARIMA
    
    sarima_model = SARIMAX(
        sarima_train,
        order=sarima_order,
        seasonal_order=seasonal_order,
        enforce_stationarity=False,
        enforce_invertibility=False
    )
    
    sarima_results = sarima_model.fit(disp=False)
    print("✅ SARIMA модель успешно обучена")
    
    # Прогноз на валидационной выборке
    val_hours = len(val_data[target].resample('H').mean())
    sarima_forecast = sarima_results.get_forecast(steps=val_hours)
    sarima_pred = sarima_forecast.predicted_mean
    
except Exception as e:
    print(f"❌ Ошибка SARIMA: {e}")
    sarima_results = None
📊 Обучение SARIMA модели...
🎯 Определение параметров SARIMA:
Сезонность: 7 дней (недельная)
Тренд: Есть
Стационарность: Нет
✅ SARIMA модель успешно обучена
print("\n📊 Обучение Prophet модели...")

from prophet import Prophet

try:
    # Подготовка данных для Prophet
    prophet_train = train_data[target].reset_index()
    prophet_train.columns = ['ds', 'y']
    prophet_train['ds'] = pd.to_datetime(prophet_train['ds'])
    
    # Создаем модель с сезонностями из вашего EDA
    prophet_model = Prophet(
        yearly_seasonality=True,    # Годовая сезонность
        weekly_seasonality=True,    # Недельная сезонность  
        daily_seasonality=True,     # Суточная сезонность
        changepoint_prior_scale=0.05
    )
    
    # Добавляем дополнительные сезонности
    prophet_model.add_seasonality(
        name='hourly', 
        period=1, 
        fourier_order=3
    )
    
    # Обучаем модель
    prophet_model.fit(prophet_train)
    print("✅ Prophet модель успешно обучена")
    
    # Создаем будущие даты для прогноза
    future_dates = prophet_model.make_future_dataframe(
        periods=len(val_data), 
        freq='1min', 
        include_history=False
    )
    
    # Прогноз
    prophet_forecast = prophet_model.predict(future_dates)
    prophet_pred = prophet_forecast['yhat'].values
    
except Exception as e:
    print(f"❌ Ошибка Prophet: {e}")
    prophet_model = None
📊 Обучение Prophet модели...
23:23:17 - cmdstanpy - INFO - Chain [1] start processing
23:25:03 - cmdstanpy - INFO - Chain [1] done processing
✅ Prophet модель успешно обучена
print("\n🧠 Обучение LSTM модели...")

from sklearn.preprocessing import MinMaxScaler
from tensorflow.keras.models import Sequential
from tensorflow.keras.layers import LSTM, Dense, Dropout
from tensorflow.keras.optimizers import Adam


def create_lstm_dataset(data, lookback=24):
    """Создание dataset для LSTM"""
    X, y = [], []
    for i in range(lookback, len(data)):
        X.append(data[i-lookback:i, 0])
        y.append(data[i, 0])
    return np.array(X), np.array(y)

try:
    # Нормализация данных
    scaler = MinMaxScaler(feature_range=(0, 1))
    scaled_data = scaler.fit_transform(train_data[target].values.reshape(-1, 1))
    
    # Создание обучающей выборки
    lookback = 24  # 24 часа истории
    X_train_lstm, y_train_lstm = create_lstm_dataset(scaled_data, lookback)
    
    # Изменение формы для LSTM [samples, time steps, features]
    X_train_lstm = X_train_lstm.reshape(X_train_lstm.shape[0], X_train_lstm.shape[1], 1)
    
    # Создание модели LSTM
    lstm_model = Sequential([
        LSTM(50, return_sequences=True, input_shape=(lookback, 1)),
        Dropout(0.2),
        LSTM(50, return_sequences=False),
        Dropout(0.2),
        Dense(25),
        Dense(1)
    ])
    
    # Компиляция модели
    lstm_model.compile(
        optimizer=Adam(learning_rate=0.001),
        loss='mse',
        metrics=['mae']
    )
    
    print("✅ LSTM модель создана, начинаем обучение...")
    
    # Обучение модели
    history = lstm_model.fit(
        X_train_lstm, y_train_lstm,
        batch_size=32,
        epochs=10,
        validation_split=0.2,
        verbose=1
    )
    
    print("✅ LSTM модель успешно обучена")
    
    # Прогноз на валидационных данных
    val_scaled = scaler.transform(val_data[target].values.reshape(-1, 1))
    X_val_lstm, y_val_lstm = create_lstm_dataset(val_scaled, lookback)
    X_val_lstm = X_val_lstm.reshape(X_val_lstm.shape[0], X_val_lstm.shape[1], 1)
    
    lstm_pred_scaled = lstm_model.predict(X_val_lstm)
    lstm_pred = scaler.inverse_transform(lstm_pred_scaled).flatten()
    
except Exception as e:
    print(f"❌ Ошибка LSTM: {e}")
    lstm_model = None
🧠 Обучение LSTM модели...
✅ LSTM модель создана, начинаем обучение...
Epoch 1/10
4320/4320 ━━━━━━━━━━━━━━━━━━━━ 47s 10ms/step - loss: 0.0014 - mae: 0.0206 - val_loss: 4.5971e-04 - val_mae: 0.0103
Epoch 2/10
4320/4320 ━━━━━━━━━━━━━━━━━━━━ 45s 10ms/step - loss: 0.0011 - mae: 0.0183 - val_loss: 5.4376e-04 - val_mae: 0.0142
Epoch 3/10
4320/4320 ━━━━━━━━━━━━━━━━━━━━ 51s 12ms/step - loss: 0.0011 - mae: 0.0177 - val_loss: 6.0263e-04 - val_mae: 0.0168
Epoch 4/10
4320/4320 ━━━━━━━━━━━━━━━━━━━━ 49s 11ms/step - loss: 0.0011 - mae: 0.0175 - val_loss: 4.1844e-04 - val_mae: 0.0075
Epoch 5/10
4320/4320 ━━━━━━━━━━━━━━━━━━━━ 49s 11ms/step - loss: 0.0011 - mae: 0.0173 - val_loss: 4.1890e-04 - val_mae: 0.0070
Epoch 6/10
4320/4320 ━━━━━━━━━━━━━━━━━━━━ 47s 11ms/step - loss: 0.0011 - mae: 0.0172 - val_loss: 4.5476e-04 - val_mae: 0.0098
Epoch 7/10
4320/4320 ━━━━━━━━━━━━━━━━━━━━ 48s 11ms/step - loss: 0.0011 - mae: 0.0170 - val_loss: 4.5936e-04 - val_mae: 0.0109
Epoch 8/10
4320/4320 ━━━━━━━━━━━━━━━━━━━━ 48s 11ms/step - loss: 0.0011 - mae: 0.0169 - val_loss: 4.1744e-04 - val_mae: 0.0066
Epoch 9/10
4320/4320 ━━━━━━━━━━━━━━━━━━━━ 48s 11ms/step - loss: 0.0011 - mae: 0.0168 - val_loss: 4.6239e-04 - val_mae: 0.0083
Epoch 10/10
4320/4320 ━━━━━━━━━━━━━━━━━━━━ 48s 11ms/step - loss: 0.0011 - mae: 0.0168 - val_loss: 4.2925e-04 - val_mae: 0.0096
✅ LSTM модель успешно обучена
1395/1395 ━━━━━━━━━━━━━━━━━━━━ 6s 4ms/step
# ДОБАВЬТЕ ПЕРЕД ОБУЧЕНИЕМ МОДЕЛЕЙ:
X_train = train_data[best_features]
y_train = train_data[target]
X_val = val_data[best_features] 
y_val = val_data[target]
X_test = test_data[best_features]
y_test = test_data[target]


# =============================================================================
# ИСПРАВЛЕНИЕ ОСНОВНЫХ ОШИБОК
# =============================================================================

print("\n🔧 ИСПРАВЛЕНИЕ ОШИБОК И ЗАПУСК ВСЕХ МОДЕЛЕЙ...")

# 1. Добавляем недостающие импорты
from tensorflow.keras.callbacks import EarlyStopping
import xgboost as xgb
from catboost import CatBoostRegressor
from lightgbm import LGBMRegressor

# 2. Подготавливаем данные для моделей
print("📊 Подготовка данных для всех моделей...")
X_train = train_data[best_features]
y_train = train_data[target]
X_val = val_data[best_features]
y_val = val_data[target]
X_test = test_data[best_features] 
y_test = test_data[target]

print(f"X_train: {X_train.shape}, y_train: {y_train.shape}")
print(f"X_val: {X_val.shape}, y_val: {y_val.shape}")

# 3. Создаем словарь для результатов
models_performance = {}

# =============================================================================
# 1. LIGHTGBM (БАЗОВАЯ МОДЕЛЬ)
# =============================================================================
print("\n💡 1. Обучение LightGBM...")
try:
    lgb_model = LGBMRegressor(
        n_estimators=100,
        max_depth=8,
        learning_rate=0.1,
        random_state=42,
        n_jobs=-1,
        verbose=-1
    )
    
    lgb_model.fit(X_train, y_train)
    lgb_pred = lgb_model.predict(X_val)
    lgb_mae = mean_absolute_error(y_val, lgb_pred)
    models_performance['LightGBM'] = lgb_mae
    print(f"✅ LightGBM: MAE = {lgb_mae:.4f} кВт")
    
except Exception as e:
    print(f"❌ LightGBM ошибка: {e}")
    models_performance['LightGBM'] = float('inf')

# =============================================================================
# 2. УЛУЧШЕННЫЙ XGBOOST
# =============================================================================
print("\n⚡ 2. Обучение XGBoost...")
try:
    xgb_model = xgb.XGBRegressor(
        n_estimators=200,
        max_depth=10,
        learning_rate=0.05,
        subsample=0.9,
        colsample_bytree=0.8,
        random_state=42,
        n_jobs=-1
    )
    
    xgb_model.fit(X_train, y_train)
    xgb_pred = xgb_model.predict(X_val)
    xgb_mae = mean_absolute_error(y_val, xgb_pred)
    models_performance['XGBoost'] = xgb_mae
    print(f"✅ XGBoost: MAE = {xgb_mae:.4f} кВт")
    
except Exception as e:
    print(f"❌ XGBoost ошибка: {e}")
    models_performance['XGBoost'] = float('inf')

# =============================================================================
# 3. CATBOOST
# =============================================================================
print("\n📈 3. Обучение CatBoost...")
try:
    catboost_model = CatBoostRegressor(
        iterations=300,
        depth=8,
        learning_rate=0.05,
        random_seed=42,
        verbose=False
    )
    
    catboost_model.fit(X_train, y_train)
    catboost_pred = catboost_model.predict(X_val)
    catboost_mae = mean_absolute_error(y_val, catboost_pred)
    models_performance['CatBoost'] = catboost_mae
    print(f"✅ CatBoost: MAE = {catboost_mae:.4f} кВт")
    
except Exception as e:
    print(f"❌ CatBoost ошибка: {e}")
    models_performance['CatBoost'] = float('inf')

# =============================================================================
# 4. TCN (БЕЗ EarlyStopping)
# =============================================================================
print("\n📊 4. Обучение TCN...")
try:
    tcn_model = create_tcn_model(lookback=24)
    tcn_model.compile(
        optimizer=Adam(learning_rate=0.001),
        loss='mse',
        metrics=['mae']
    )
    
    print("✅ TCN архитектура создана, начинаем обучение...")
    
    # Обучение без EarlyStopping
    history_tcn = tcn_model.fit(
        X_train_lstm, y_train_lstm,
        batch_size=64,
        epochs=10,  # Уменьшаем эпохи
        validation_split=0.2,
        verbose=1
    )
    
    # Прогноз
    tcn_pred_scaled = tcn_model.predict(X_val_lstm, verbose=0)
    tcn_pred = scaler.inverse_transform(tcn_pred_scaled).flatten()
    
    # Вычисляем MAE
    val_target = val_data[target][lookback:lookback+len(tcn_pred)]
    tcn_mae = mean_absolute_error(val_target, tcn_pred)
    models_performance['TCN'] = tcn_mae
    print(f"✅ TCN: MAE = {tcn_mae:.4f} кВт")
    
except Exception as e:
    print(f"❌ TCN ошибка: {e}")
    models_performance['TCN'] = float('inf')

# =============================================================================
# 5. GRU (БЕЗ EarlyStopping)
# =============================================================================
print("\n🔄 5. Обучение GRU...")
try:
    gru_model = create_gru_model(lookback=24)
    gru_model.compile(
        optimizer=Adam(learning_rate=0.001),
        loss='mse',
        metrics=['mae']
    )
    
    print("✅ GRU архитектура создана, начинаем обучение...")
    
    # Обучение без EarlyStopping
    history_gru = gru_model.fit(
        X_train_lstm, y_train_lstm,
        batch_size=64,
        epochs=10,  # Уменьшаем эпохи
        validation_split=0.2,
        verbose=1
    )
    
    # Прогноз
    gru_pred_scaled = gru_model.predict(X_val_lstm, verbose=0)
    gru_pred = scaler.inverse_transform(gru_pred_scaled).flatten()
    
    # Вычисляем MAE
    val_target = val_data[target][lookback:lookback+len(gru_pred)]
    gru_mae = mean_absolute_error(val_target, gru_pred)
    models_performance['GRU'] = gru_mae
    print(f"✅ GRU: MAE = {gru_mae:.4f} кВт")
    
except Exception as e:
    print(f"❌ GRU ошибка: {e}")
    models_performance['GRU'] = float('inf')

# =============================================================================
# 6. УЛУЧШЕННЫЙ LIGHTGBM
# =============================================================================
print("\n💡 6. Обучение улучшенного LightGBM...")
try:
    lgb_improved = LGBMRegressor(
        n_estimators=300,
        max_depth=12,
        learning_rate=0.03,
        num_leaves=128,
        subsample=0.9,
        colsample_bytree=0.8,
        reg_alpha=0.1,
        reg_lambda=0.1,
        random_state=42,
        n_jobs=-1,
        verbose=-1
    )
    
    lgb_improved.fit(X_train, y_train)
    lgb_improved_pred = lgb_improved.predict(X_val)
    lgb_improved_mae = mean_absolute_error(y_val, lgb_improved_pred)
    models_performance['LightGBM_Improved'] = lgb_improved_mae
    print(f"✅ LightGBM Improved: MAE = {lgb_improved_mae:.4f} кВт")
    
except Exception as e:
    print(f"❌ LightGBM Improved ошибка: {e}")
    models_performance['LightGBM_Improved'] = float('inf')

# =============================================================================
# 7. ENSEMBLE МОДЕЛЬ
# =============================================================================
print("\n🤝 7. Создание ансамблевой модели...")
try:
    # Собираем прогнозы успешных моделей
    ensemble_predictions = []
    model_names = []
    
    # Добавляем успешные модели
    for model_name in ['LightGBM', 'XGBoost', 'CatBoost', 'LightGBM_Improved']:
        if model_name in models_performance and models_performance[model_name] != float('inf'):
            if model_name == 'LightGBM':
                ensemble_predictions.append(lgb_pred)
            elif model_name == 'XGBoost':
                ensemble_predictions.append(xgb_pred)
            elif model_name == 'CatBoost':
                ensemble_predictions.append(catboost_pred)
            elif model_name == 'LightGBM_Improved':
                ensemble_predictions.append(lgb_improved_pred)
            model_names.append(model_name)
    
    if len(ensemble_predictions) >= 2:
        # Усредняем прогнозы
        ensemble_pred = np.mean(ensemble_predictions, axis=0)
        ensemble_mae = mean_absolute_error(y_val, ensemble_pred)
        models_performance['Ensemble'] = ensemble_mae
        print(f"✅ Ensemble ({len(ensemble_predictions)} моделей): MAE = {ensemble_mae:.4f} кВт")
    else:
        print("❌ Недостаточно моделей для ансамбля")
        models_performance['Ensemble'] = float('inf')
        
except Exception as e:
    print(f"❌ Ensemble ошибка: {e}")
    models_performance['Ensemble'] = float('inf')

# =============================================================================
# ВЫВОД РЕЗУЛЬТАТОВ
# =============================================================================
print("\n" + "="*70)
print("🎯 ИТОГОВЫЕ РЕЗУЛЬТАТЫ ВСЕХ МОДЕЛЕЙ")
print("="*70)

# Сортируем модели по качеству
sorted_models = sorted([(m, ma) for m, ma in models_performance.items() if ma != float('inf')], 
                      key=lambda x: x[1])

print("\n🏆 РЕЙТИНГ МОДЕЛЕЙ:")
print("-" * 60)
print(f"{'Модель':<20} {'MAE (кВт)':<12} {'Улучшение':<15}")
print("-" * 60)

for i, (model, mae) in enumerate(sorted_models):
    if i == 0:
        best_mae = mae
        print(f"{model:<20} {mae:<12.4f} {'🏆 ЛУЧШАЯ':<15}")
    else:
        improvement = ((best_mae - mae) / best_mae * 100)
        print(f"{model:<20} {mae:<12.4f} {improvement:+.1f}%{' ':<10}")

print("-" * 60)

best_model_name = sorted_models[0][0]
best_model_mae = sorted_models[0][1]

print(f"\n🎉 ЛУЧШАЯ МОДЕЛЬ: {best_model_name} (MAE: {best_model_mae:.4f} кВт)")

# Сравнение с предыдущим LightGBM
if 'LightGBM' in models_performance:
    old_lightgbm_mae = 0.1077
    improvement_vs_old = ((old_lightgbm_mae - best_model_mae) / old_lightgbm_mae * 100)
    print(f"📊 Сравнение с предыдущим LightGBM (0.1077 кВт): {improvement_vs_old:+.1f}%")

print("\n" + "="*70)
🔧 ИСПРАВЛЕНИЕ ОШИБОК И ЗАПУСК ВСЕХ МОДЕЛЕЙ...
📊 Подготовка данных для всех моделей...
X_train: (172800, 20), y_train: (172800,)
X_val: (44640, 20), y_val: (44640,)

💡 1. Обучение LightGBM...
✅ LightGBM: MAE = 0.1066 кВт

⚡ 2. Обучение XGBoost...
✅ XGBoost: MAE = 0.1173 кВт

📈 3. Обучение CatBoost...
✅ CatBoost: MAE = 0.1083 кВт

📊 4. Обучение TCN...
✅ TCN архитектура создана, начинаем обучение...
Epoch 1/10
2160/2160 ━━━━━━━━━━━━━━━━━━━━ 20s 8ms/step - loss: 0.0206 - mae: 0.0601 - val_loss: 0.0012 - val_mae: 0.0266
Epoch 2/10
2160/2160 ━━━━━━━━━━━━━━━━━━━━ 18s 8ms/step - loss: 0.0027 - mae: 0.0334 - val_loss: 0.0011 - val_mae: 0.0268
Epoch 3/10
2160/2160 ━━━━━━━━━━━━━━━━━━━━ 17s 8ms/step - loss: 0.0021 - mae: 0.0285 - val_loss: 0.0012 - val_mae: 0.0294
Epoch 4/10
2160/2160 ━━━━━━━━━━━━━━━━━━━━ 17s 8ms/step - loss: 0.0021 - mae: 0.0285 - val_loss: 8.0571e-04 - val_mae: 0.0221
Epoch 5/10
2160/2160 ━━━━━━━━━━━━━━━━━━━━ 17s 8ms/step - loss: 0.0018 - mae: 0.0259 - val_loss: 0.0011 - val_mae: 0.0281
Epoch 6/10
2160/2160 ━━━━━━━━━━━━━━━━━━━━ 17s 8ms/step - loss: 0.0017 - mae: 0.0250 - val_loss: 0.0017 - val_mae: 0.0360
Epoch 7/10
2160/2160 ━━━━━━━━━━━━━━━━━━━━ 17s 8ms/step - loss: 0.0016 - mae: 0.0241 - val_loss: 0.0013 - val_mae: 0.0309
Epoch 8/10
2160/2160 ━━━━━━━━━━━━━━━━━━━━ 17s 8ms/step - loss: 0.0015 - mae: 0.0240 - val_loss: 0.0011 - val_mae: 0.0266
Epoch 9/10
2160/2160 ━━━━━━━━━━━━━━━━━━━━ 18s 8ms/step - loss: 0.0015 - mae: 0.0233 - val_loss: 0.0013 - val_mae: 0.0302
Epoch 10/10
2160/2160 ━━━━━━━━━━━━━━━━━━━━ 16s 8ms/step - loss: 0.0014 - mae: 0.0230 - val_loss: 0.0014 - val_mae: 0.0326
✅ TCN: MAE = 0.3410 кВт

🔄 5. Обучение GRU...
✅ GRU архитектура создана, начинаем обучение...
Epoch 1/10
2160/2160 ━━━━━━━━━━━━━━━━━━━━ 72s 31ms/step - loss: 0.0027 - mae: 0.0310 - val_loss: 5.7932e-04 - val_mae: 0.0111
Epoch 2/10
2160/2160 ━━━━━━━━━━━━━━━━━━━━ 64s 30ms/step - loss: 0.0013 - mae: 0.0203 - val_loss: 6.0722e-04 - val_mae: 0.0148
Epoch 3/10
2160/2160 ━━━━━━━━━━━━━━━━━━━━ 63s 29ms/step - loss: 0.0012 - mae: 0.0187 - val_loss: 5.0125e-04 - val_mae: 0.0124
Epoch 4/10
2160/2160 ━━━━━━━━━━━━━━━━━━━━ 63s 29ms/step - loss: 0.0011 - mae: 0.0177 - val_loss: 0.0012 - val_mae: 0.0299
Epoch 5/10
2160/2160 ━━━━━━━━━━━━━━━━━━━━ 68s 32ms/step - loss: 0.0011 - mae: 0.0169 - val_loss: 0.0019 - val_mae: 0.0284
Epoch 6/10
2160/2160 ━━━━━━━━━━━━━━━━━━━━ 65s 30ms/step - loss: 0.0010 - mae: 0.0163 - val_loss: 6.2318e-04 - val_mae: 0.0126
Epoch 7/10
2160/2160 ━━━━━━━━━━━━━━━━━━━━ 65s 30ms/step - loss: 9.9052e-04 - mae: 0.0161 - val_loss: 4.9652e-04 - val_mae: 0.0118
Epoch 8/10
2160/2160 ━━━━━━━━━━━━━━━━━━━━ 63s 29ms/step - loss: 9.6825e-04 - mae: 0.0158 - val_loss: 5.2632e-04 - val_mae: 0.0127
Epoch 9/10
2160/2160 ━━━━━━━━━━━━━━━━━━━━ 63s 29ms/step - loss: 9.4861e-04 - mae: 0.0156 - val_loss: 5.1405e-04 - val_mae: 0.0144
Epoch 10/10
2160/2160 ━━━━━━━━━━━━━━━━━━━━ 60s 28ms/step - loss: 9.1975e-04 - mae: 0.0153 - val_loss: 5.2914e-04 - val_mae: 0.0119
✅ GRU: MAE = 0.1595 кВт

💡 6. Обучение улучшенного LightGBM...
✅ LightGBM Improved: MAE = 0.1061 кВт

🤝 7. Создание ансамблевой модели...
✅ Ensemble (4 моделей): MAE = 0.1065 кВт

======================================================================
🎯 ИТОГОВЫЕ РЕЗУЛЬТАТЫ ВСЕХ МОДЕЛЕЙ
======================================================================

🏆 РЕЙТИНГ МОДЕЛЕЙ:
------------------------------------------------------------
Модель               MAE (кВт)    Улучшение      
------------------------------------------------------------
LightGBM_Improved    0.1061       🏆 ЛУЧШАЯ       
Ensemble             0.1065       -0.3%          
LightGBM             0.1066       -0.5%          
CatBoost             0.1083       -2.1%          
XGBoost              0.1173       -10.5%          
GRU                  0.1595       -50.2%          
TCN                  0.3410       -221.3%          
------------------------------------------------------------

🎉 ЛУЧШАЯ МОДЕЛЬ: LightGBM_Improved (MAE: 0.1061 кВт)
📊 Сравнение с предыдущим LightGBM (0.1077 кВт): +1.5%

======================================================================
print("\n🔧 Исправление ошибок и корректное сравнение моделей...")

models_performance = {}

# 1. SARIMA - исправляем размерности
if sarima_results is not None:
    try:
        # SARIMA прогнозировал почасовые данные, приводим к минутным
        sarima_pred_minutes = sarima_pred.repeat(60)[:len(val_data)]  # Повторяем каждый час 60 раз
        sarima_mae = mean_absolute_error(val_data[target], sarima_pred_minutes)
        models_performance['SARIMA'] = sarima_mae
        print(f"✅ SARIMA: MAE = {sarima_mae:.4f} кВт")
    except Exception as e:
        print(f"❌ SARIMA ошибка: {e}")

# 2. Prophet - проверяем размерности
if prophet_model is not None:
    try:
        # Prophet прогнозирует на те же даты, что и val_data
        prophet_mae = mean_absolute_error(val_data[target][:len(prophet_pred)], prophet_pred)
        models_performance['Prophet'] = prophet_mae
        print(f"✅ Prophet: MAE = {prophet_mae:.4f} кВт")
    except Exception as e:
        print(f"❌ Prophet ошибка: {e}")

# 3. LSTM - уже правильные размерности
if lstm_model is not None:
    try:
        # LSTM прогноз начинается с lookback-го элемента
        lstm_mae = mean_absolute_error(val_data[target][lookback:lookback+len(lstm_pred)], lstm_pred)
        models_performance['LSTM'] = lstm_mae
        print(f"✅ LSTM: MAE = {lstm_mae:.4f} кВт")
    except Exception as e:
        print(f"❌ LSTM ошибка: {e}")

# Выводим итоговые результаты
print("\n🎯 ИТОГОВЫЕ РЕЗУЛЬТАТЫ НА ВАЛИДАЦИОННОЙ ВЫБОРКЕ:")
for model, mae in sorted(models_performance.items(), key=lambda x: x[1]):
    print(f"  {model}: MAE = {mae:.4f} кВт")

if models_performance:
    best_model_name = min(models_performance, key=models_performance.get)
    print(f"\n🏆 ЛУЧШАЯ МОДЕЛЬ: {best_model_name} (MAE: {models_performance[best_model_name]:.4f} кВт)")
    
    # Сравнение с предыдущими результатами
    print("\n📊 СРАВНЕНИЕ С ПРЕДЫДУЩИМИ РЕЗУЛЬТАТАМИ:")
    print(f"LightGBM (старый): MAE = 0.1077 кВт")
    print(f"{best_model_name} (новый): MAE = {models_performance[best_model_name]:.4f} кВт")
    
    improvement = ((0.1077 - models_performance[best_model_name]) / 0.1077) * 100
    print(f"Улучшение: {improvement:+.1f}%")
else:
    print("❌ Ни одна модель не дала результатов")
🔧 Исправление ошибок и корректное сравнение моделей...
✅ SARIMA: MAE = 0.7353 кВт
✅ Prophet: MAE = 4.9924 кВт
✅ LSTM: MAE = 0.1407 кВт

🎯 ИТОГОВЫЕ РЕЗУЛЬТАТЫ НА ВАЛИДАЦИОННОЙ ВЫБОРКЕ:
  LSTM: MAE = 0.1407 кВт
  SARIMA: MAE = 0.7353 кВт
  Prophet: MAE = 4.9924 кВт

🏆 ЛУЧШАЯ МОДЕЛЬ: LSTM (MAE: 0.1407 кВт)

📊 СРАВНЕНИЕ С ПРЕДЫДУЩИМИ РЕЗУЛЬТАТАМИ:
LightGBM (старый): MAE = 0.1077 кВт
LSTM (новый): MAE = 0.1407 кВт
Улучшение: -30.7%
print(f"\n🎯 Финальное тестирование лучшей модели ({best_model_name})...")

# Тестируем лучшую модель на тестовой выборке
# [Здесь будет код тестирования лучшей модели]

# Сравнение с вашими предыдущими результатами
print("\n📊 СРАВНЕНИЕ С ПРЕДЫДУЩИМИ РЕЗУЛЬТАТАМИ:")
print(f"LightGBM (старый): MAE = 0.1077 кВт")
print(f"{best_model_name} (новый): MAE = {models_performance[best_model_name]:.4f} кВт")

improvement = ((0.1077 - models_performance[best_model_name]) / 0.1077) * 100
print(f"Улучшение: {improvement:+.1f}%")
🎯 Финальное тестирование лучшей модели (LSTM)...

📊 СРАВНЕНИЕ С ПРЕДЫДУЩИМИ РЕЗУЛЬТАТАМИ:
LightGBM (старый): MAE = 0.1077 кВт
LSTM (новый): MAE = 0.1407 кВт
Улучшение: -30.7%
# ВИЗУАЛИЗАЦИЯ ПРОГНОЗОВ
print("\n📈 Визуализация прогнозов...")

plt.figure(figsize=(15, 10))

# Берем первые 100 точек для наглядности
sample_size = 100
sample_start = 0

# График 1: Сравнение всех моделей
plt.subplot(2, 2, 1)
plt.plot(val_data[target].iloc[sample_start:sample_start+sample_size].values, 
         label='Факт', linewidth=2, color='black')

if 'SARIMA' in models_performance:
    plt.plot(sarima_pred_minutes[sample_start:sample_start+sample_size], 
             label='SARIMA', alpha=0.7)
    
if 'Prophet' in models_performance:
    plt.plot(prophet_pred[sample_start:sample_start+sample_size], 
             label='Prophet', alpha=0.7)
    
if 'LSTM' in models_performance:
    # LSTM начинается с lookback, поэтому сдвигаем
    lstm_start = lookback + sample_start
    lstm_end = lstm_start + min(sample_size, len(lstm_pred) - sample_start)
    if lstm_end > lstm_start:
        plt.plot(range(sample_start, sample_start + (lstm_end - lstm_start)), 
                 lstm_pred[sample_start:sample_start + (lstm_end - lstm_start)], 
                 label='LSTM', alpha=0.7)

plt.title('Сравнение прогнозов моделей')
plt.xlabel('Временные точки')
plt.ylabel('Нагрузка (кВт)')
plt.legend()
plt.grid(True, alpha=0.3)

# График 2: Ошибки прогнозирования
plt.subplot(2, 2, 2)
if models_performance:
    models_names = list(models_performance.keys())
    mae_values = [models_performance[model] for model in models_names]
    
    colors = ['green' if model == best_model_name else 'gray' for model in models_names]
    bars = plt.bar(models_names, mae_values, color=colors, alpha=0.7)
    
    plt.title('Сравнение MAE моделей')
    plt.ylabel('MAE (кВт)')
    plt.grid(True, alpha=0.3)
    
    # Добавляем значения на столбцы
    for bar, mae in zip(bars, mae_values):
        plt.text(bar.get_x() + bar.get_width()/2, bar.get_height() + 0.001,
                f'{mae:.4f}', ha='center', va='bottom')

# График 3: Распределение ошибок лучшей модели
plt.subplot(2, 2, 3)
if best_model_name == 'SARIMA' and 'SARIMA' in models_performance:
    errors = sarima_pred_minutes - val_data[target].values
elif best_model_name == 'Prophet' and 'Prophet' in models_performance:
    errors = prophet_pred - val_data[target][:len(prophet_pred)].values
elif best_model_name == 'LSTM' and 'LSTM' in models_performance:
    errors = lstm_pred - val_data[target][lookback:lookback+len(lstm_pred)].values
else:
    errors = np.array([])

if len(errors) > 0:
    plt.hist(errors, bins=50, alpha=0.7, color='orange', edgecolor='black')
    plt.axvline(x=0, color='red', linestyle='--', alpha=0.7)
    plt.title(f'Распределение ошибок ({best_model_name})')
    plt.xlabel('Ошибка прогноза (кВт)')
    plt.ylabel('Частота')
    plt.grid(True, alpha=0.3)

# График 4: Качество прогнозов по времени суток
plt.subplot(2, 2, 4)
if len(errors) > 0:
    # Берем часы из валидационной выборки
    val_hours = val_data.index.hour[lookback:lookback+len(errors)] if best_model_name == 'LSTM' else val_data.index.hour[:len(errors)]
    
    hourly_errors = pd.DataFrame({
        'hour': val_hours,
        'abs_error': np.abs(errors)
    }).groupby('hour')['abs_error'].mean()
    
    plt.bar(hourly_errors.index, hourly_errors.values, alpha=0.7, color='purple')
    plt.title('Средняя ошибка по часам суток')
    plt.xlabel('Час дня')
    plt.ylabel('Средняя абсолютная ошибка (кВт)')
    plt.xticks(range(0, 24))
    plt.grid(True, alpha=0.3)

plt.tight_layout()
plt.show()

print("✅ Визуализация завершена!")
📈 Визуализация прогнозов...

✅ Визуализация завершена!
print("📊 ДЕТАЛЬНЫЙ АНАЛИЗ РЕЗУЛЬТАТОВ ГРАФИКОВ (ИСПРАВЛЕННАЯ ВЕРСИЯ)")
print("=" * 70)

# 1. АНАЛИЗ СРАВНЕНИЯ МОДЕЛЕЙ
print("\n🎯 1. СРАВНЕНИЕ МОДЕЛЕЙ НА ВАЛИДАЦИОННОЙ ВЫБОРКЕ")
print("-" * 50)

models_comparison = pd.DataFrame({
    'Модель': list(models_performance.keys()),
    'MAE (кВт)': list(models_performance.values()),
    'Улучшение vs LightGBM (%)': [((0.1077 - mae) / 0.1077 * 100) for mae in models_performance.values()]
}).sort_values('MAE (кВт)')

print(models_comparison.to_string(index=False))

print(f"\n📈 ЛУЧШАЯ МОДЕЛЬ: {best_model_name}")
print(f"📊 MAE: {models_performance[best_model_name]:.4f} кВт")
print(f"🎯 Разница с LightGBM: {((0.1077 - models_performance[best_model_name]) / 0.1077 * 100):+.1f}%")

# 2. АНАЛИЗ ПРОГНОЗОВ НА ПЕРВЫЕ 20 ТОЧЕК (ИСПРАВЛЕННАЯ ВЕРСИЯ)
print("\n\n📈 2. АНАЛИЗ ПРОГНОЗОВ (первые 20 точек)")
print("-" * 70)

# Создаем DataFrame с правильными индексами
forecast_data = []

# Берем первые 20 точек из валидационной выборки
for i in range(20):
    time_point = val_data.index[i]
    actual = val_data[target].iloc[i]
    
    row = {'Время': time_point, 'Факт (кВт)': actual}
    
    # Добавляем прогнозы моделей если они есть
    if 'SARIMA' in models_performance and i < len(sarima_pred_minutes):
        row['SARIMA (кВт)'] = sarima_pred_minutes[i]
    
    if 'Prophet' in models_performance and i < len(prophet_pred):
        row['Prophet (кВт)'] = prophet_pred[i]
    
    if 'LSTM' in models_performance:
        # LSTM начинается с lookback, поэтому сдвигаем индексы
        lstm_idx = i
        if lstm_idx < len(lstm_pred):
            row['LSTM (кВт)'] = lstm_pred[lstm_idx]
    
    forecast_data.append(row)

forecast_comparison = pd.DataFrame(forecast_data)
print(forecast_comparison.to_string(index=False))

# 3. ДЕТАЛЬНЫЙ АНАЛИЗ ОШИБОК ЛУЧШЕЙ МОДЕЛИ
print("\n\n📊 3. ДЕТАЛЬНЫЙ АНАЛИЗ ОШИБОК ЛУЧШЕЙ МОДЕЛИ (LSTM)")
print("-" * 70)

# Для LSTM вычисляем ошибки
errors = lstm_pred - val_data[target][lookback:lookback+len(lstm_pred)].values
actual_values = val_data[target][lookback:lookback+len(lstm_pred)].values

error_analysis = pd.DataFrame({
    'Метрика': [
        'Средняя абсолютная ошибка (MAE)',
        'Средняя ошибка (Bias)',
        'Стандартное отклонение ошибок',
        'Максимальная положительная ошибка',
        'Максимальная отрицательная ошибка',
        'Медианная абсолютная ошибка',
        'Процент ошибок < 0.1 кВт',
        'Процент ошибок < 0.05 кВт',
        'Процент ошибок < 0.2 кВт'
    ],
    'Значение': [
        np.mean(np.abs(errors)),
        np.mean(errors),
        np.std(errors),
        np.max(errors),
        np.min(errors),
        np.median(np.abs(errors)),
        (np.abs(errors) < 0.1).mean() * 100,
        (np.abs(errors) < 0.05).mean() * 100,
        (np.abs(errors) < 0.2).mean() * 100
    ],
    'Единица измерения': [
        'кВт', 'кВт', 'кВт', 'кВт', 'кВт', 'кВт', '%', '%', '%'
    ]
})

print(error_analysis.to_string(index=False))

# 4. АНАЛИЗ ОШИБОК ПО ЧАСАМ СУТОК
print("\n\n⏰ 4. АНАЛИЗ ОШИБОК ПО ЧАСАМ СУТОК")
print("-" * 70)

hours = val_data.index.hour[lookback:lookback+len(errors)]

hourly_analysis = []
for hour in range(24):
    hour_mask = hours == hour
    if hour_mask.sum() > 0:
        hour_errors = errors[hour_mask]
        hour_actual = actual_values[hour_mask]
        
        hourly_analysis.append({
            'Час': hour,
            'Ср. ошибка (кВт)': np.mean(np.abs(hour_errors)),
            'Стд. ошибки': np.std(hour_errors),
            'Кол-во': hour_mask.sum(),
            'Ср. потребление (кВт)': np.mean(hour_actual)
        })

hourly_df = pd.DataFrame(hourly_analysis).round(4)
print(hourly_df.to_string(index=False))

# 5. АНАЛИЗ САМЫХ БОЛЬШИХ ОШИБОК
print("\n\n⚠️  5. ТОП-10 САМЫХ БОЛЬШИХ ОШИБОК LSTM")
print("-" * 70)

# Находим индексы самых больших ошибок
top_error_indices = np.argsort(np.abs(errors))[-10:][::-1]

worst_errors = []
for i, idx in enumerate(top_error_indices):
    time_idx = lookback + idx
    timestamp = val_data.index[time_idx]
    
    worst_errors.append({
        'Место': i + 1,
        'Время': timestamp,
        'Факт (кВт)': actual_values[idx],
        'Прогноз LSTM (кВт)': lstm_pred[idx],
        'Ошибка (кВт)': errors[idx],
        'Абс. ошибка (кВт)': np.abs(errors[idx])
    })

worst_errors_df = pd.DataFrame(worst_errors)
print(worst_errors_df.to_string(index=False))

# 6. СРАВНЕНИЕ РАСПРЕДЕЛЕНИЙ ПОТРЕБЛЕНИЯ
print("\n\n📈 6. СРАВНЕНИЕ РАСПРЕДЕЛЕНИЙ ПОТРЕБЛЕНИЯ")
print("-" * 70)

distribution_stats = pd.DataFrame({
    'Выборка': ['Обучающая', 'Валидационная', 'Тестовая'],
    'Записей': [len(train_data), len(val_data), len(test_data)],
    'Среднее (кВт)': [
        train_data[target].mean(),
        val_data[target].mean(), 
        test_data[target].mean()
    ],
    'Медиана (кВт)': [
        train_data[target].median(),
        val_data[target].median(),
        test_data[target].median()
    ],
    'Стд. (кВт)': [
        train_data[target].std(),
        val_data[target].std(),
        test_data[target].std()
    ]
})

print(distribution_stats.to_string(index=False))

# 7. АНАЛИЗ КАЧЕСТВА ПРОГНОЗОВ ПО УРОВНЯМ ПОТРЕБЛЕНИЯ
print("\n\n📊 7. АНАЛИЗ КАЧЕСТВА ПРОГНОЗОВ ПО УРОВНЯМ ПОТРЕБЛЕНИЯ")
print("-" * 70)

# Создаем уровни потребления
consumption_levels = pd.cut(actual_values, 
                           bins=[0, 0.5, 1.0, 2.0, 5.0, 10.0],
                           labels=['Очень низкое', 'Низкое', 'Среднее', 'Высокое', 'Очень высокое'])

level_analysis = []
for level in consumption_levels.categories:
    level_mask = consumption_levels == level
    if level_mask.sum() > 0:
        level_errors = errors[level_mask]
        level_actual = actual_values[level_mask]
        
        level_analysis.append({
            'Уровень потребления': level,
            'Кол-во точек': level_mask.sum(),
            'Ср. MAE (кВт)': np.mean(np.abs(level_errors)),
            'Ср. потребление (кВт)': np.mean(level_actual),
            'Доля ошибок > 0.1 кВт': (np.abs(level_errors) > 0.1).mean() * 100
        })

level_df = pd.DataFrame(level_analysis).round(4)
print(level_df.to_string(index=False))

print("\n" + "=" * 70)
print("🎯 ИТОГОВАЯ ОЦЕНКА РЕЗУЛЬТАТОВ")
print("=" * 70)
📊 ДЕТАЛЬНЫЙ АНАЛИЗ РЕЗУЛЬТАТОВ ГРАФИКОВ (ИСПРАВЛЕННАЯ ВЕРСИЯ)
======================================================================

🎯 1. СРАВНЕНИЕ МОДЕЛЕЙ НА ВАЛИДАЦИОННОЙ ВЫБОРКЕ
--------------------------------------------------
 Модель  MAE (кВт)  Улучшение vs LightGBM (%)
   LSTM   0.140737                 -30.675482
 SARIMA   0.735293                -582.723228
Prophet   4.992408               -4535.476350

📈 ЛУЧШАЯ МОДЕЛЬ: LSTM
📊 MAE: 0.1407 кВт
🎯 Разница с LightGBM: -30.7%


📈 2. АНАЛИЗ ПРОГНОЗОВ (первые 20 точек)
----------------------------------------------------------------------
              Время  Факт (кВт)  SARIMA (кВт)  Prophet (кВт)  LSTM (кВт)
2007-05-01 00:00:00       0.340      0.199145      -0.336109    0.269540
2007-05-01 00:01:00       0.294      0.199145      -0.341728    0.288220
2007-05-01 00:02:00       0.294      0.199145      -0.347281    0.289780
2007-05-01 00:03:00       0.294      0.199145      -0.352770    0.280694
2007-05-01 00:04:00       0.292      0.199145      -0.358194    0.271236
2007-05-01 00:05:00       0.292      0.199145      -0.363555    0.268632
2007-05-01 00:06:00       0.384      0.199145      -0.368851    0.269644
2007-05-01 00:07:00       0.368      0.199145      -0.374084    0.268081
2007-05-01 00:08:00       0.366      0.199145      -0.379253    0.271239
2007-05-01 00:09:00       0.364      0.199145      -0.384359    0.270526
2007-05-01 00:10:00       0.360      0.199145      -0.389401    0.325560
2007-05-01 00:11:00       0.360      0.199145      -0.394381    0.378693
2007-05-01 00:12:00       0.290      0.199145      -0.399299    0.346089
2007-05-01 00:13:00       0.270      0.199145      -0.404154    0.345935
2007-05-01 00:14:00       0.268      0.199145      -0.408948    0.359710
2007-05-01 00:15:00       0.268      0.199145      -0.413679    0.374246
2007-05-01 00:16:00       0.268      0.199145      -0.418350    0.373954
2007-05-01 00:17:00       0.268      0.199145      -0.422960    0.369178
2007-05-01 00:18:00       0.266      0.199145      -0.427510    0.365556
2007-05-01 00:19:00       0.266      0.199145      -0.431999    0.363656


📊 3. ДЕТАЛЬНЫЙ АНАЛИЗ ОШИБОК ЛУЧШЕЙ МОДЕЛИ (LSTM)
----------------------------------------------------------------------
                          Метрика  Значение Единица измерения
  Средняя абсолютная ошибка (MAE)  0.140737               кВт
            Средняя ошибка (Bias)  0.031946               кВт
    Стандартное отклонение ошибок  0.299049               кВт
Максимальная положительная ошибка  3.194857               кВт
Максимальная отрицательная ошибка -3.923609               кВт
      Медианная абсолютная ошибка  0.068672               кВт
         Процент ошибок < 0.1 кВт 76.719114                 %
        Процент ошибок < 0.05 кВт 30.928367                 %
         Процент ошибок < 0.2 кВт 86.612426                 %


⏰ 4. АНАЛИЗ ОШИБОК ПО ЧАСАМ СУТОК
----------------------------------------------------------------------
 Час  Ср. ошибка (кВт)  Стд. ошибки  Кол-во  Ср. потребление (кВт)
   0            0.1040       0.2067    1836                 0.5330
   1            0.0875       0.1364    1860                 0.3888
   2            0.0861       0.1349    1860                 0.3432
   3            0.0836       0.1296    1860                 0.3250
   4            0.0851       0.1343    1860                 0.3206
   5            0.0886       0.1444    1860                 0.3213
   6            0.1679       0.3976    1860                 0.7145
   7            0.1498       0.3210    1860                 1.1608
   8            0.1382       0.3276    1860                 1.3810
   9            0.0961       0.2218    1860                 1.2529
  10            0.0981       0.2055    1860                 0.9123
  11            0.1183       0.2506    1860                 0.8796
  12            0.1472       0.3351    1860                 0.9830
  13            0.1795       0.3533    1860                 1.0050
  14            0.1780       0.3568    1860                 1.1032
  15            0.1527       0.3054    1860                 0.9695
  16            0.1383       0.2818    1860                 0.8210
  17            0.1551       0.3239    1860                 0.9129
  18            0.1683       0.3592    1860                 1.1315
  19            0.2304       0.4469    1860                 1.5325
  20            0.2303       0.4317    1860                 1.8202
  21            0.2287       0.4091    1860                 2.2547
  22            0.1632       0.3208    1860                 1.6871
  23            0.1021       0.2038    1860                 0.9099


⚠️  5. ТОП-10 САМЫХ БОЛЬШИХ ОШИБОК LSTM
----------------------------------------------------------------------
 Место               Время  Факт (кВт)  Прогноз LSTM (кВт)  Ошибка (кВт)  Абс. ошибка (кВт)
     1 2007-05-25 06:30:00       4.596            0.672391     -3.923609           3.923609
     2 2007-05-14 07:01:00       5.270            1.666260     -3.603740           3.603740
     3 2007-05-01 18:49:00       4.892            1.376645     -3.515355           3.515355
     4 2007-05-08 12:34:00       6.264            2.809695     -3.454305           3.454305
     5 2007-05-16 06:36:00       4.752            1.338722     -3.413278           3.413278
     6 2007-05-06 08:07:00       6.126            2.734025     -3.391975           3.391975
     7 2007-05-03 06:30:00       4.378            1.086241     -3.291759           3.291759
     8 2007-05-17 19:15:00       5.624            2.386744     -3.237256           3.237256
     9 2007-05-31 06:37:00       4.500            1.300879     -3.199121           3.199121
    10 2007-05-28 14:00:00       0.782            3.976857      3.194857           3.194857


📈 6. СРАВНЕНИЕ РАСПРЕДЕЛЕНИЙ ПОТРЕБЛЕНИЯ
----------------------------------------------------------------------
      Выборка  Записей  Среднее (кВт)  Медиана (кВт)  Стд. (кВт)
    Обучающая   172800       1.282679          0.776    1.242927
Валидационная    44640       0.985862          0.462    1.006413
     Тестовая    43200       0.826553          0.356    0.952562


📊 7. АНАЛИЗ КАЧЕСТВА ПРОГНОЗОВ ПО УРОВНЯМ ПОТРЕБЛЕНИЯ
----------------------------------------------------------------------
Уровень потребления  Кол-во точек  Ср. MAE (кВт)  Ср. потребление (кВт)  Доля ошибок > 0.1 кВт
       Очень низкое         23283         0.0815                 0.2922                11.7597
             Низкое          3773         0.1379                 0.6838                26.0005
            Среднее         12108         0.1184                 1.4424                23.0096
            Высокое          5188         0.4254                 3.0180                70.0463
      Очень высокое           264         0.8366                 5.6721                93.9394

======================================================================
🎯 ИТОГОВАЯ ОЦЕНКА РЕЗУЛЬТАТОВ
======================================================================
# =============================================================================
# ВИЗУАЛИЗАЦИЯ ВСЕХ РЕЗУЛЬТАТОВ
# =============================================================================
print("\n" + "="*70)
print("🎯 ИТОГОВЫЕ РЕЗУЛЬТАТЫ ВСЕХ МОДЕЛЕЙ")
print("="*70)

# Сортируем модели по качеству
sorted_models = sorted(models_performance.items(), key=lambda x: x[1] if x[1] != float('inf') else float('inf'))

print("\n🏆 РЕЙТИНГ МОДЕЛЕЙ:")
print("-" * 60)
print(f"{'Модель':<20} {'MAE (кВт)':<12} {'Улучшение vs LightGBM':<20}")
print("-" * 60)

for model, mae in sorted_models:
    if mae != float('inf'):
        improvement = ((0.1077 - mae) / 0.1077 * 100)
        improvement_str = f"{improvement:+.1f}%"
        
        if model == 'LightGBM':
            print(f"{model:<20} {mae:<12.4f} {'→ БАЗА':<20} 🏆")
        elif improvement > 0:
            print(f"{model:<20} {mae:<12.4f} {improvement_str:<20} ✅")
        else:
            print(f"{model:<20} {mae:<12.4f} {improvement_str:<20} ❌")
    else:
        print(f"{model:<20} {'ОШИБКА':<12} {'-':<20} 💀")

print("-" * 60)

# Находим лучшую модель
best_model_name = sorted_models[0][0]
best_model_mae = sorted_models[0][1]

print(f"\n🎉 ЛУЧШАЯ МОДЕЛЬ: {best_model_name} (MAE: {best_model_mae:.4f} кВт)")

if best_model_name != 'LightGBM':
    improvement = ((0.1077 - best_model_mae) / 0.1077 * 100)
    print(f"🎯 УЛУЧШЕНИЕ: {improvement:+.2f}%")
else:
    print("🎯 LightGBM остается непревзойденным!")

# ВИЗУАЛИЗАЦИЯ
plt.figure(figsize=(16, 10))

# График 1: Сравнение всех моделей
plt.subplot(2, 2, 1)
models_to_plot = [model for model, mae in sorted_models if mae != float('inf')][:8]  # Топ-8
mae_values = [models_performance[model] for model in models_to_plot]

colors = ['green' if model == best_model_name else 
          'lightgreen' if 'LightGBM' in model else
          'orange' if improvement > 0 else 'gray' 
          for model in models_to_plot]

bars = plt.bar(models_to_plot, mae_values, color=colors, alpha=0.7)
plt.title('Сравнение MAE всех моделей', fontsize=14, fontweight='bold')
plt.ylabel('MAE (кВт)')
plt.xticks(rotation=45, ha='right')
plt.grid(True, alpha=0.3)

# Добавляем значения на столбцы
for bar, mae in zip(bars, mae_values):
    plt.text(bar.get_x() + bar.get_width()/2, bar.get_height() + 0.001,
             f'{mae:.4f}', ha='center', va='bottom', fontsize=9)

# График 2: Улучшение относительно LightGBM
plt.subplot(2, 2, 2)
improvements = [((0.1077 - mae) / 0.1077 * 100) for mae in mae_values]

colors_imp = ['green' if imp > 0 else 'red' for imp in improvements]
bars_imp = plt.bar(models_to_plot, improvements, color=colors_imp, alpha=0.7)
plt.title('Улучшение vs LightGBM (%)', fontsize=14, fontweight='bold')
plt.ylabel('Улучшение (%)')
plt.xticks(rotation=45, ha='right')
plt.grid(True, alpha=0.3)

# Добавляем значения на столбцы
for bar, imp in zip(bars_imp, improvements):
    plt.text(bar.get_x() + bar.get_width()/2, bar.get_height() + (0.5 if imp > 0 else -1),
             f'{imp:+.1f}%', ha='center', va='bottom' if imp > 0 else 'top', fontsize=9)

# График 3: Прогнозы лучших 3 моделей
plt.subplot(2, 2, 3)
top_3_models = models_to_plot[:3]

# Берем первые 50 точек для наглядности
sample_size = 50
plt.plot(y_val.values[:sample_size], label='Факт', linewidth=2, color='black')

for model in top_3_models:
    if model == 'LightGBM':
        pred = y_val_lgb[:sample_size]
    elif model == 'LSTM':
        pred = lstm_pred[:sample_size]
    elif model == 'TCN':
        pred = tcn_pred[:sample_size]
    elif model == 'GRU':
        pred = gru_pred[:sample_size]
    elif model == 'XGBoost_TS':
        pred = xgb_pred[:sample_size]
    elif model == 'CatBoost':
        pred = catboost_pred[:sample_size]
    elif model == 'LightGBM_Improved':
        pred = lgb_improved_pred[:sample_size]
    elif model == 'Ensemble':
        pred = ensemble_pred[:sample_size]
    else:
        continue
        
    plt.plot(pred[:sample_size], label=model, alpha=0.8)

plt.title('Прогнозы лучших 3 моделей', fontsize=14, fontweight='bold')
plt.xlabel('Временные точки')
plt.ylabel('Нагрузка (кВт)')
plt.legend()
plt.grid(True, alpha=0.3)

# График 4: Время обучения vs Качество (если есть данные)
plt.subplot(2, 2, 4)
# Заглушка для времени обучения (в реальности нужно замерять)
training_times = [1.0, 2.5, 3.0, 2.8, 1.2, 1.5, 1.8, 0.5][:len(mae_values)]  # примерные значения

scatter = plt.scatter(mae_values, training_times[:len(mae_values)], 
                     s=100, alpha=0.6, c=range(len(mae_values)), cmap='viridis')

plt.xlabel('MAE (кВт)')
plt.ylabel('Время обучения (отн. ед.)')
plt.title('Качество vs Время обучения', fontsize=14, fontweight='bold')
plt.grid(True, alpha=0.3)

# Добавляем подписи точек
for i, model in enumerate(models_to_plot):
    plt.annotate(model, (mae_values[i], training_times[i]), 
                xytext=(5, 5), textcoords='offset points', fontsize=8)

plt.tight_layout()
plt.show()
======================================================================
🎯 ИТОГОВЫЕ РЕЗУЛЬТАТЫ ВСЕХ МОДЕЛЕЙ
======================================================================

🏆 РЕЙТИНГ МОДЕЛЕЙ:
------------------------------------------------------------
Модель               MAE (кВт)    Улучшение vs LightGBM
------------------------------------------------------------
LSTM                 0.1407       -30.7%               ❌
SARIMA               0.7353       -582.7%              ❌
Prophet              4.9924       -4535.5%             ❌
------------------------------------------------------------

🎉 ЛУЧШАЯ МОДЕЛЬ: LSTM (MAE: 0.1407 кВт)
🎯 УЛУЧШЕНИЕ: -30.68%

# ТЕСТИРОВАНИЕ ЛУЧШЕЙ МОДЕЛИ НА ТЕСТОВОЙ ВЫБОРКЕ
print(f"\n🎯 Тестирование лучшей модели ({best_model_name}) на тестовой выборке...")

if best_model_name == 'LSTM':
    # Для LSTM - прогнозируем на тестовых данных
    test_scaled = scaler.transform(test_data[target].values.reshape(-1, 1))
    X_test_lstm, y_test_lstm = create_lstm_dataset(test_scaled, lookback)
    X_test_lstm = X_test_lstm.reshape(X_test_lstm.shape[0], X_test_lstm.shape[1], 1)
    
    test_pred_scaled = lstm_model.predict(X_test_lstm)
    test_pred = scaler.inverse_transform(test_pred_scaled).flatten()
    
    test_mae = mean_absolute_error(test_data[target][lookback:lookback+len(test_pred)], test_pred)
    
elif best_model_name == 'Prophet':
    # Для Prophet - создаем будущие даты для теста
    test_future = prophet_model.make_future_dataframe(
        periods=len(test_data), 
        freq='1min', 
        include_history=False
    )
    test_forecast = prophet_model.predict(test_future)
    test_pred = test_forecast['yhat'].values
    test_mae = mean_absolute_error(test_data[target][:len(test_pred)], test_pred)
    
elif best_model_name == 'SARIMA':
    # Для SARIMA - прогнозируем тестовый период
    test_hours = len(test_data[target].resample('H').mean())
    test_forecast = sarima_results.get_forecast(steps=test_hours)
    test_pred_hourly = test_forecast.predicted_mean
    test_pred = test_pred_hourly.repeat(60)[:len(test_data)]  # Приводим к минутным данным
    test_mae = mean_absolute_error(test_data[target], test_pred)

print(f"📊 Результаты на тестовой выборке:")
print(f"MAE = {test_mae:.4f} кВт")

# ФИНАЛЬНОЕ СРАВНЕНИЕ
print("\n" + "="*60)
print("ФИНАЛЬНЫЕ РЕЗУЛЬТАТЫ ПЕРЕОБУЧЕНИЯ")
print("="*60)
print(f"🏆 Лучшая модель: {best_model_name}")
print(f"📊 MAE на валидации: {models_performance[best_model_name]:.4f} кВт")
print(f"📊 MAE на тесте: {test_mae:.4f} кВт")
print(f"📊 LightGBM (старый): 0.1077 кВт")

final_improvement = ((0.1077 - test_mae) / 0.1077) * 100
print(f"🎯 Улучшение на тесте: {final_improvement:+.1f}%")

if final_improvement > 0:
    print("✅ ПЕРЕОБУЧЕНИЕ УСПЕШНО! Новые модели показывают лучшие результаты!")
else:
    print("⚠️  Новые модели показывают схожие результаты со старыми")

print("="*60)
🎯 Тестирование лучшей модели (LSTM) на тестовой выборке...
1350/1350 ━━━━━━━━━━━━━━━━━━━━ 5s 3ms/step
📊 Результаты на тестовой выборке:
MAE = 0.1360 кВт

============================================================
ФИНАЛЬНЫЕ РЕЗУЛЬТАТЫ ПЕРЕОБУЧЕНИЯ
============================================================
🏆 Лучшая модель: LSTM
📊 MAE на валидации: 0.1407 кВт
📊 MAE на тесте: 0.1360 кВт
📊 LightGBM (старый): 0.1077 кВт
🎯 Улучшение на тесте: -26.3%
⚠️  Новые модели показывают схожие результаты со старыми
============================================================
# # ЭКСПЕРТНАЯ ОЦЕНКА РЕЗУЛЬТАТОВ
# print("\n🔍 ЭКСПЕРТНАЯ ОЦЕНКА:")

# # Оценка качества моделей
# def evaluate_model_performance(mae):
#     if mae < 0.05:
#         return "ОТЛИЧНО 🏆", "Модель практически идеальна"
#     elif mae < 0.1:
#         return "ОЧЕНЬ ХОРОШО ✅", "Высокая точность прогнозирования"
#     elif mae < 0.15:
#         return "ХОРОШО 👍", "Удовлетворительная точность"
#     elif mae < 0.2:
#         return "УДОВЛЕТВОРИТЕЛЬНО ⚠️", "Приемлемая точность"
#     else:
#         return "ПЛОХО ❌", "Требует значительного улучшения"

# print("\n📊 ОЦЕНКА КАЧЕСТВА МОДЕЛЕЙ:")
# for model, mae in models_performance.items():
#     rating, comment = evaluate_model_performance(mae)
#     print(f"  {model}: {rating} (MAE: {mae:.4f} кВт) - {comment}")

# # Оценка LSTM vs LightGBM
# lstm_mae = models_performance.get('LSTM', 1.0)
# lightgbm_mae = 0.1077
# difference = ((lightgbm_mae - lstm_mae) / lightgbm_mae) * 100

# print(f"\n⚔️  СРАВНЕНИЕ LSTM vs LightGBM:")
# print(f"  LightGBM: {lightgbm_mae:.4f} кВт")
# print(f"  LSTM: {lstm_mae:.4f} кВт")
# print(f"  Разница: {difference:+.2f}%")

# if abs(difference) < 5:
#     print("  📍 ВЫВОД: Модели показывают СХОЖЕЕ качество")
# elif difference > 0:
#     print("  📍 ВЫВОД: LightGBM ЛУЧШЕ на {:.1f}%".format(abs(difference)))
# else:
#     print("  📍 ВЫВОД: LSTM ЛУЧШЕ на {:.1f}%".format(abs(difference)))

# # Анализ стабильности ошибок
# error_std = np.std(errors)
# mean_abs_error = np.mean(np.abs(errors))

# print(f"\n📏 АНАЛИЗ СТАБИЛЬНОСТИ ОШИБОК:")
# print(f"  Средняя абсолютная ошибка: {mean_abs_error:.4f} кВт")
# print(f"  Стандартное отклонение ошибок: {error_std:.4f} кВт")
# print(f"  Коэффициент вариации: {(error_std/mean_abs_error*100):.1f}%")

# if error_std < mean_abs_error * 0.5:
#     print("  ✅ Ошибки СТАБИЛЬНЫЕ - модель предсказуема")
# else:
#     print("  ⚠️  Ошибки НЕСТАБИЛЬНЫЕ - модель непредсказуема")

# # Анализ практической применимости
# print(f"\n💼 ПРАКТИЧЕСКАЯ ПРИМЕНИМОСТЬ:")
# print(f"  Точность прогноза: {(1 - mean_abs_error/val_data[target].mean())*100:.1f}%")
# print(f"  Средняя ошибка в ваттах: {mean_abs_error*1000:.0f} Вт")

# if mean_abs_error < 0.1:
#     print("  ✅ Модель готова к практическому применению")
#     print("  🎯 Можно использовать для:")
#     print("     • Оптимизации энергопотребления")
#     print("     • Прогнозирования пиковых нагрузок") 
#     print("     • Планирования энергозатрат")
# else:
#     print("  ⚠️  Требуется доработка для практического применения")

# print(f"\n📈 РЕКОМЕНДАЦИИ:")
# if best_model_name == 'LSTM' and abs(difference) < 2:
#     print("  1. Используйте LightGBM - проще в интерпретации и обслуживании")
#     print("  2. LSTM оставьте как исследовательский подход")
# elif best_model_name == 'LSTM' and difference < -2:
#     print("  1. LSTM показала лучшие результаты - используйте её")
#     print("  2. Увеличьте количество эпох обучения для улучшения")
# else:
#     print("  1. LightGBM остается лучшим выбором")
#     print("  2. Сфокусируйтесь на улучшении признаков")

# print("\n" + "=" * 70)
# print("🎓 ВЫВОД ДЛЯ КУРСОВОЙ:")
# print("=" * 70)
# print("✅ LightGBM с качественными временными признаками показывает")
# print("   state-of-the-art результаты для прогнозирования энергопотребления")
# print("✅ Ваш feature engineering - ключевой фактор успеха")
# print("✅ Модель готова к практическому применению с точностью ~90%")
# print("✅ Результаты научно обоснованы и воспроизводимы")
# =============================================================================
# ФИНАЛЬНЫЙ КОМПЛЕКСНЫЙ АНАЛИЗ ВСЕХ МОДЕЛЕЙ ВРЕМЕННЫХ РЯДОВ
# =============================================================================

print("\n" + "="*80)
print("🎯 ФИНАЛЬНЫЙ КОМПЛЕКСНЫЙ АНАЛИЗ ВСЕХ МОДЕЛЕЙ ВРЕМЕННЫХ РЯДОВ")
print("="*80)

# 1. КЛАССИФИКАЦИЯ МОДЕЛЕЙ ПО ТИПАМ
print("\n📊 КЛАССИФИКАЦИЯ МОДЕЛЕЙ ПО ТИПАМ:")
print("-" * 60)

model_categories = {
    'Tree-based ML': ['LightGBM', 'LightGBM_Improved', 'XGBoost', 'CatBoost'],
    'Нейросетевые (RNN)': ['LSTM', 'GRU'],
    'Нейросетевые (CNN)': ['TCN', 'TCN_Improved'],
    'Ансамбли': ['Ensemble'],
    'Классические TS': ['SARIMA', 'Prophet']
}

# Фильтруем только существующие модели
existing_models = list(models_performance.keys())
categorized_models = {}

for category, models in model_categories.items():
    existing_in_category = [m for m in models if m in existing_models]
    if existing_in_category:
        categorized_models[category] = existing_in_category
        print(f"  {category}: {', '.join(existing_in_category)}")

# 2. ДЕТАЛЬНАЯ ОЦЕНКА КАЧЕСТВА ВСЕХ МОДЕЛЕЙ
print(f"\n📈 ДЕТАЛЬНАЯ ОЦЕНКА КАЧЕСТВА ВСЕХ {len(existing_models)} МОДЕЛЕЙ:")
print("-" * 80)

def comprehensive_model_evaluation(mae, model_name):
    """Комплексная оценка модели"""
    if mae < 0.05:
        return "ОТЛИЧНО 🏆", "Практически идеальная точность", "✅ Готова к продакшену"
    elif mae < 0.08:
        return "ОЧЕНЬ ХОРОШО ✅", "Высокая точность", "✅ Готова к продакшену"
    elif mae < 0.1:
        return "ХОРОШО 👍", "Удовлетворительная точность", "✅ Приемлема для продакшена"
    elif mae < 0.12:
        return "УДОВЛЕТВОРИТЕЛЬНО ⚠️", "Приемлемая точность", "⚠️ Требует доработки"
    elif mae < 0.15:
        return "НИЖЕ СРЕДНЕГО 📉", "Недостаточная точность", "❌ Не готова для продакшена"
    else:
        return "ПЛОХО ❌", "Требует значительного улучшения", "❌ Непригодна"

# Создаем DataFrame для детального анализа
analysis_data = []
for model in existing_models:
    mae = models_performance[model]
    rating, precision_comment, readiness = comprehensive_model_evaluation(mae, model)
    
    # Определяем категорию модели
    category = next((cat for cat, models in categorized_models.items() if model in models), "Другое")
    
    # Сравнение с LightGBM (базовой моделью)
    lightgbm_mae = 0.1077
    improvement = ((lightgbm_mae - mae) / lightgbm_mae * 100)
    
    analysis_data.append({
        'Модель': model,
        'Категория': category,
        'MAE (кВт)': mae,
        'Оценка': rating,
        'Точность': precision_comment,
        'Готовность': readiness,
        'Улучшение vs LightGBM (%)': improvement
    })

# Сортируем по MAE (от лучшей к худшей)
analysis_df = pd.DataFrame(analysis_data).sort_values('MAE (кВт)')
print(analysis_df.to_string(index=False, float_format='%.4f'))

# 3. СТАТИСТИЧЕСКИЙ АНАЛИЗ РЕЗУЛЬТАТОВ
print(f"\n📊 СТАТИСТИЧЕСКИЙ АНАЛИЗ РЕЗУЛЬТАТОВ:")
print("-" * 60)

mae_values = [models_performance[model] for model in existing_models]
print(f"• Количество моделей: {len(existing_models)}")
print(f"• Лучший MAE: {min(mae_values):.4f} кВт ({best_model_name})")
print(f"• Худший MAE: {max(mae_values):.4f} кВт")
print(f"• Средний MAE: {np.mean(mae_values):.4f} кВт")
print(f"• Медианный MAE: {np.median(mae_values):.4f} кВт")
print(f"• Стандартное отклонение: {np.std(mae_values):.4f} кВт")

# 4. АНАЛИЗ ПО КАТЕГОРИЯМ МОДЕЛЕЙ
print(f"\n🎯 АНАЛИЗ ЭФФЕКТИВНОСТИ ПО КАТЕГОРИЯМ:")
print("-" * 60)

category_performance = {}
for category, models in categorized_models.items():
    if models:
        category_mae = [models_performance[model] for model in models]
        best_in_category = min(category_mae)
        avg_in_category = np.mean(category_mae)
        category_performance[category] = {
            'Лучшая модель': models[np.argmin(category_mae)],
            'Лучший MAE': best_in_category,
            'Средний MAE': avg_in_category,
            'Кол-во моделей': len(models)
        }

for category, stats in category_performance.items():
    improvement = ((0.1077 - stats['Лучший MAE']) / 0.1077 * 100)
    print(f"  {category}:")
    print(f"    • Лучшая модель: {stats['Лучшая модель']} (MAE: {stats['Лучший MAE']:.4f} кВт)")
    print(f"    • Улучшение vs LightGBM: {improvement:+.2f}%")
    print(f"    • Средний MAE в категории: {stats['Средний MAE']:.4f} кВт")
    print(f"    • Количество моделей: {stats['Кол-во моделей']}")

# 5. ДЕТАЛЬНОЕ СРАВНЕНИЕ ТОП-5 МОДЕЛЕЙ
print(f"\n🏆 ДЕТАЛЬНОЕ СРАВНЕНИЕ ТОП-5 МОДЕЛЕЙ:")
print("-" * 70)

top_5_models = analysis_df.head(5)
for i, (_, row) in enumerate(top_5_models.iterrows()):
    print(f"{i+1}. {row['Модель']}:")
    print(f"   • Категория: {row['Категория']}")
    print(f"   • MAE: {row['MAE (кВт)']:.4f} кВт")
    print(f"   • Оценка: {row['Оценка']}")
    print(f"   • Готовность: {row['Готовность']}")
    print(f"   • Улучшение vs LightGBM: {row['Улучшение vs LightGBM (%)']:+.2f}%")

# 6. АНАЛИЗ СТАБИЛЬНОСТИ И ПРЕДСКАЗУЕМОСТИ
print(f"\n📏 УГЛУБЛЕННЫЙ АНАЛИЗ СТАБИЛЬНОСТИ:")
print("-" * 60)

# Анализ ошибок лучшей модели
best_model_predictions = None
if best_model_name == 'LightGBM':
    best_model_predictions = lgb_pred
elif best_model_name == 'LSTM':
    best_model_predictions = lstm_pred
elif best_model_name == 'LightGBM_Improved':
    best_model_predictions = lgb_improved_pred
elif best_model_name == 'Ensemble':
    best_model_predictions = ensemble_pred

if best_model_predictions is not None:
    errors = best_model_predictions - y_val.values[:len(best_model_predictions)]
    
    print(f"Анализ ошибок лучшей модели ({best_model_name}):")
    print(f"  • Средняя абсолютная ошибка: {np.mean(np.abs(errors)):.4f} кВт")
    print(f"  • Средняя ошибка (bias): {np.mean(errors):.4f} кВт")
    print(f"  • Стандартное отклонение: {np.std(errors):.4f} кВт")
    print(f"  • Максимальная ошибка: {np.max(np.abs(errors)):.4f} кВт")
    print(f"  • Коэффициент вариации: {(np.std(errors)/np.mean(np.abs(errors))*100):.1f}%")
    
    # Анализ распределения ошибок
    error_percentiles = np.percentile(np.abs(errors), [25, 50, 75, 90, 95])
    print(f"  • 25% перцентиль: {error_percentiles[0]:.4f} кВт")
    print(f"  • Медиана: {error_percentiles[1]:.4f} кВт")
    print(f"  • 75% перцентиль: {error_percentiles[2]:.4f} кВт")
    print(f"  • 90% перцентиль: {error_percentiles[3]:.4f} кВт")
    
    # Оценка стабильности
    cv = (np.std(errors) / np.mean(np.abs(errors))) * 100
    if cv < 50:
        stability = "ОТЛИЧНАЯ ✅"
    elif cv < 100:
        stability = "ХОРОШАЯ 👍" 
    elif cv < 150:
        stability = "УДОВЛЕТВОРИТЕЛЬНАЯ ⚠️"
    else:
        stability = "НИЗКАЯ ❌"
    
    print(f"  • Стабильность прогнозов: {stability} (CV: {cv:.1f}%)")

# 7. ПРАКТИЧЕСКАЯ ПРИМЕНИМОСТЬ И РЕКОМЕНДАЦИИ
print(f"\n💼 ПРАКТИЧЕСКАЯ ПРИМЕНИМОСТЬ И РЕКОМЕНДАЦИИ:")
print("-" * 60)

best_mae = analysis_df.iloc[0]['MAE (кВт)']
accuracy = (1 - best_mae / y_val.mean()) * 100
error_watts = best_mae * 1000

print(f"ЛУЧШАЯ МОДЕЛЬ: {best_model_name}")
print(f"• Точность прогнозирования: {accuracy:.1f}%")
print(f"• Средняя ошибка: {error_watts:.0f} Вт")
print(f"• Качество: {analysis_df.iloc[0]['Оценка']}")

if best_mae < 0.1:
    print("🎯 ВЫВОД: Модель готова к промышленному применению")
    print("   Применение:")
    print("   • Оптимизация энергопотребления в реальном времени")
    print("   • Прогнозирование пиковых нагрузок")
    print("   • Планирование энергозатрат")
    print("   • Обнаружение аномалий потребления")
else:
    print("⚠️  ВЫВОД: Требуется доработка для промышленного применения")

# 8. ФИНАЛЬНЫЕ РЕКОМЕНДАЦИИ ПО ВЫБОРУ МОДЕЛИ
print(f"\n🎓 ФИНАЛЬНЫЕ РЕКОМЕНДАЦИИ ДЛЯ КУРСОВОЙ РАБОТЫ:")
print("-" * 60)

# Определяем рекомендации на основе анализа
if best_model_name in ['LightGBM', 'LightGBM_Improved', 'XGBoost', 'CatBoost']:
    print("✅ РЕКОМЕНДАЦИЯ: Использовать Tree-based модель")
    print("   Преимущества:")
    print("   • Высокая точность и скорость")
    print("   • Хорошая интерпретируемость")
    print("   • Простота развертывания")
    print("   • Устойчивость к шумам")
    
elif best_model_name in ['LSTM', 'GRU', 'TCN', 'TCN_Improved']:
    print("✅ РЕКОМЕНДАЦИЯ: Использовать нейросетевую модель") 
    print("   Преимущества:")
    print("   • Учет сложных временных зависимостей")
    print("   • Автоматическое извлечение признаков")
    print("   • Потенциал для дальнейшего улучшения")
    
elif best_model_name == 'Ensemble':
    print("✅ РЕКОМЕНДАЦИЯ: Использовать ансамблевую модель")
    print("   Преимущества:")
    print("   • Максимальная точность")
    print("   • Устойчивость к переобучению")
    print("   • Комбинирование сильных сторон разных подходов")

print(f"\n🎯 ОБОСНОВАНИЕ ВЫБОРА {best_model_name}:")
print(f"   • Наилучший MAE: {best_mae:.4f} кВт")
print(f"   • Улучшение vs базовой модели: {((0.1077 - best_mae) / 0.1077 * 100):+.2f}%")
print(f"   • Готовность: {analysis_df.iloc[0]['Готовность']}")
print(f"   • Категория: {analysis_df.iloc[0]['Категория']}")

# 9. ВЫВОДЫ ДЛЯ НАУЧНОЙ РАБОТЫ
print(f"\n🔬 ВЫВОДЫ ДЛЯ КУРСОВОЙ РАБОТЫ:")
print("-" * 60)

print("1. ПРОВЕДЕНО КОМПЛЕКСНОЕ ИССЛЕДОВАНИЕ:")
print(f"   • Протестировано {len(existing_models)} различных моделей временных рядов")
print("   • Охвачены все основные подходы: tree-based, нейросетевые, ансамбли")

print("2. КЛЮЧЕВЫЕ НАУЧНЫЕ РЕЗУЛЬТАТЫ:")
print(f"   • Наилучшие результаты показала модель: {best_model_name}")
print(f"   • Достигнута точность прогнозирования: {accuracy:.1f}%")
print(f"   • Подтверждена эффективность feature engineering")

print("3. ПРАКТИЧЕСКАЯ ЗНАЧИМОСТЬ:")
print("   • Разработана готовая к применению модель прогнозирования")
print("   • Определены оптимальные подходы для доменного потребления")
print("   • Создан воспроизводимый pipeline для подобных задач")

print("4. ПЕРСПЕКТИВЫ ДАЛЬНЕЙШИХ ИССЛЕДОВАНИЙ:")
print("   • Использование трансформеров для временных рядов")
print("   • Применение transfer learning между доменами")
print("   • Интеграция внешних факторов (погода, события)")

print("\n" + "="*80)
print("🏆 ФИНАЛЬНЫЙ ВЕРДИКТ:")
print("="*80)
print(f"ОПТИМАЛЬНАЯ МОДЕЛЬ ДЛЯ ПРОГНОЗИРОВАНИЯ ЭНЕРГОПОТРЕБЛЕНИЯ:")
print(f"🎯 {best_model_name} (MAE: {best_mae:.4f} кВт, Точность: {accuracy:.1f}%)")
print("="*80)
================================================================================
🎯 ФИНАЛЬНЫЙ КОМПЛЕКСНЫЙ АНАЛИЗ ВСЕХ МОДЕЛЕЙ ВРЕМЕННЫХ РЯДОВ
================================================================================

📊 КЛАССИФИКАЦИЯ МОДЕЛЕЙ ПО ТИПАМ:
------------------------------------------------------------
  Нейросетевые (RNN): LSTM
  Классические TS: SARIMA, Prophet

📈 ДЕТАЛЬНАЯ ОЦЕНКА КАЧЕСТВА ВСЕХ 3 МОДЕЛЕЙ:
--------------------------------------------------------------------------------
 Модель          Категория  MAE (кВт)          Оценка                        Точность                 Готовность  Улучшение vs LightGBM (%)
   LSTM Нейросетевые (RNN)     0.1407 НИЖЕ СРЕДНЕГО 📉          Недостаточная точность ❌ Не готова для продакшена                   -30.6755
 SARIMA    Классические TS     0.7353         ПЛОХО ❌ Требует значительного улучшения               ❌ Непригодна                  -582.7232
Prophet    Классические TS     4.9924         ПЛОХО ❌ Требует значительного улучшения               ❌ Непригодна                 -4535.4764

📊 СТАТИСТИЧЕСКИЙ АНАЛИЗ РЕЗУЛЬТАТОВ:
------------------------------------------------------------
• Количество моделей: 3
• Лучший MAE: 0.1407 кВт (LSTM)
• Худший MAE: 4.9924 кВт
• Средний MAE: 1.9561 кВт
• Медианный MAE: 0.7353 кВт
• Стандартное отклонение: 2.1606 кВт

🎯 АНАЛИЗ ЭФФЕКТИВНОСТИ ПО КАТЕГОРИЯМ:
------------------------------------------------------------
  Нейросетевые (RNN):
    • Лучшая модель: LSTM (MAE: 0.1407 кВт)
    • Улучшение vs LightGBM: -30.68%
    • Средний MAE в категории: 0.1407 кВт
    • Количество моделей: 1
  Классические TS:
    • Лучшая модель: SARIMA (MAE: 0.7353 кВт)
    • Улучшение vs LightGBM: -582.72%
    • Средний MAE в категории: 2.8639 кВт
    • Количество моделей: 2

🏆 ДЕТАЛЬНОЕ СРАВНЕНИЕ ТОП-5 МОДЕЛЕЙ:
----------------------------------------------------------------------
1. LSTM:
   • Категория: Нейросетевые (RNN)
   • MAE: 0.1407 кВт
   • Оценка: НИЖЕ СРЕДНЕГО 📉
   • Готовность: ❌ Не готова для продакшена
   • Улучшение vs LightGBM: -30.68%
2. SARIMA:
   • Категория: Классические TS
   • MAE: 0.7353 кВт
   • Оценка: ПЛОХО ❌
   • Готовность: ❌ Непригодна
   • Улучшение vs LightGBM: -582.72%
3. Prophet:
   • Категория: Классические TS
   • MAE: 4.9924 кВт
   • Оценка: ПЛОХО ❌
   • Готовность: ❌ Непригодна
   • Улучшение vs LightGBM: -4535.48%

📏 УГЛУБЛЕННЫЙ АНАЛИЗ СТАБИЛЬНОСТИ:
------------------------------------------------------------
Анализ ошибок лучшей модели (LSTM):
  • Средняя абсолютная ошибка: 0.4952 кВт
  • Средняя ошибка (bias): 0.0319 кВт
  • Стандартное отклонение: 0.8772 кВт
  • Максимальная ошибка: 6.3168 кВт
  • Коэффициент вариации: 177.1%
  • 25% перцентиль: 0.0706 кВт
  • Медиана: 0.1609 кВт
  • 75% перцентиль: 0.6812 кВт
  • 90% перцентиль: 1.5370 кВт
  • Стабильность прогнозов: НИЗКАЯ ❌ (CV: 177.1%)

💼 ПРАКТИЧЕСКАЯ ПРИМЕНИМОСТЬ И РЕКОМЕНДАЦИИ:
------------------------------------------------------------
ЛУЧШАЯ МОДЕЛЬ: LSTM
• Точность прогнозирования: 85.7%
• Средняя ошибка: 141 Вт
• Качество: НИЖЕ СРЕДНЕГО 📉
⚠️  ВЫВОД: Требуется доработка для промышленного применения

🎓 ФИНАЛЬНЫЕ РЕКОМЕНДАЦИИ ДЛЯ КУРСОВОЙ РАБОТЫ:
------------------------------------------------------------
✅ РЕКОМЕНДАЦИЯ: Использовать нейросетевую модель
   Преимущества:
   • Учет сложных временных зависимостей
   • Автоматическое извлечение признаков
   • Потенциал для дальнейшего улучшения

🎯 ОБОСНОВАНИЕ ВЫБОРА LSTM:
   • Наилучший MAE: 0.1407 кВт
   • Улучшение vs базовой модели: -30.68%
   • Готовность: ❌ Не готова для продакшена
   • Категория: Нейросетевые (RNN)

🔬 ВЫВОДЫ ДЛЯ КУРСОВОЙ РАБОТЫ:
------------------------------------------------------------
1. ПРОВЕДЕНО КОМПЛЕКСНОЕ ИССЛЕДОВАНИЕ:
   • Протестировано 3 различных моделей временных рядов
   • Охвачены все основные подходы: tree-based, нейросетевые, ансамбли
2. КЛЮЧЕВЫЕ НАУЧНЫЕ РЕЗУЛЬТАТЫ:
   • Наилучшие результаты показала модель: LSTM
   • Достигнута точность прогнозирования: 85.7%
   • Подтверждена эффективность feature engineering
3. ПРАКТИЧЕСКАЯ ЗНАЧИМОСТЬ:
   • Разработана готовая к применению модель прогнозирования
   • Определены оптимальные подходы для доменного потребления
   • Создан воспроизводимый pipeline для подобных задач
4. ПЕРСПЕКТИВЫ ДАЛЬНЕЙШИХ ИССЛЕДОВАНИЙ:
   • Использование трансформеров для временных рядов
   • Применение transfer learning между доменами
   • Интеграция внешних факторов (погода, события)

================================================================================
🏆 ФИНАЛЬНЫЙ ВЕРДИКТ:
================================================================================
ОПТИМАЛЬНАЯ МОДЕЛЬ ДЛЯ ПРОГНОЗИРОВАНИЯ ЭНЕРГОПОТРЕБЛЕНИЯ:
🎯 LSTM (MAE: 0.1407 кВт, Точность: 85.7%)
================================================================================
# Анализ топ-3 моделей
print(f"\n🏆 ТОП-3 МОДЕЛИ:")
for i, (model, mae) in enumerate(sorted_models[:3]):
    improvement = ((0.1077 - mae) / 0.1077 * 100)
    print(f"{i+1}. {model}: MAE = {mae:.4f} кВт ({improvement:+.2f}%)")

print(f"\n💡 РЕКОМЕНДАЦИИ:")
if best_model_name == 'LightGBM':
    print("✅ LightGBM остается лучшим выбором - оптимальное соотношение")
    print("   точности, скорости и интерпретируемости")
elif 'Ensemble' in best_model_name:
    print("✅ Ансамбль показал лучший результат - используйте комбинацию моделей")
elif 'LSTM' in best_model_name or 'GRU' in best_model_name or 'TCN' in best_model_name:
    print("✅ Нейросетевая модель превзошла LightGBM - рассмотрите её использование")
    print("   для максимальной точности (ценой вычислительной сложности)")
else:
    print("✅ Другая модель показала лучший результат - проанализируйте её")
    print("   преимущества для вашей конкретной задачи")

print(f"\n🎯 ДЛЯ КУРСОВОЙ РАБОТЫ:")
print(f"1. Основная модель: {best_model_name}")
print(f"2. Сравнение с {len(models_performance)} различными подходами")
print(f"3. Научное обоснование выбора модели")
print(f"4. Детальный анализ результатов и ошибок")

print("\n" + "="*70)
🏆 ТОП-3 МОДЕЛИ:
1. LSTM: MAE = 0.1407 кВт (-30.68%)
2. SARIMA: MAE = 0.7353 кВт (-582.72%)
3. Prophet: MAE = 4.9924 кВт (-4535.48%)

💡 РЕКОМЕНДАЦИИ:
✅ Нейросетевая модель превзошла LightGBM - рассмотрите её использование
   для максимальной точности (ценой вычислительной сложности)

🎯 ДЛЯ КУРСОВОЙ РАБОТЫ:
1. Основная модель: LSTM
2. Сравнение с 3 различными подходами
3. Научное обоснование выбора модели
4. Детальный анализ результатов и ошибок

======================================================================